# Trustworthiness-Annealer: Collapse verhindern, Rate sparen

Ein AUTOENCODIX-Annealer, der β nicht nach einer festen Kurve, sondern nach einer **Preservation-Nebenbedingung** führt:

$$\max_\beta \; \beta \quad \text{so dass} \quad \min_i \tilde m_i \ge 0, \qquad \tilde m_i = \frac{m_i(X,\mu) - \text{Ziel}_i}{\text{Toleranz}_i}$$

Das ist eine ε-Constraint-Formulierung der Rate-Preservation-Abwägung; mit mehreren Metriken wird die Nebenbedingung zur
Tchebycheff-/Min-max-Bedingung (die schlechteste normierte Metrik entscheidet). β ist der Lagrange-Multiplikator, der
Controller macht den Dual-Schritt (verwandt: GECO, Rezende & Viola 2018; ControlVAE, Shao et al. 2020). Der Annealer ist
allgemein für eine oder mehrere Metriken geschrieben, **hier wird nur Trustworthiness verwendet**.

Fragen dieses Notebooks:

1. **Collapse:** Verhindert der Annealer Posterior-Collapse zuverlässig (über Seeds), und ab welchem Sollwert nicht mehr?
2. **Rate:** Wie viel Rate wird gegenüber β = 0 eingespart, ohne dass der Sollwert verletzt wird?
3. **Preservation unter Druck:** Bleibt Preservation bei wenig Informationskosten hoch und fällt erst bei hohen Kosten
   (Knick) oder fällt sie stetig mit der Rate? Gemessen über mehrere k (Trustworthiness, R_NX).
4. **AUC:** Wie folgen Krebs-AUC und Geschlechts-AUC (nur innerhalb gemischter Krebsarten) auf z?
5. **Raum:** Wie verändert sich der Latentraum von getrennten Clustern zu Overlap (labelfrei, `cluster_overlap.py`)?
6. **k des Controllers:** Wie stark hängen Stopp, Rate und Preservation davon ab, mit welchem k gesteuert wird?

Rate, AUC und Labels steuern nichts; der Controller sieht nur X und μ des festen Validierungs-Subsets.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Callable
import gc
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from sklearn.manifold import trustworthiness
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "beta_vae_experiments").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if os.environ.get("PYTHONHASHSEED") != "2":
    raise RuntimeError("TCGA braucht PYTHONHASHSEED=2 vor dem Kernel-Start: Kernel 'VAE Beta Control (TCGA)' wählen.")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import autoencodix as acx
from autoencodix.utils._annealer import AnnealingScheduler
from autoencodix.losses.varix_loss import VarixLoss
from zadu.measures import local_continuity_meta_criteria

from beta_vae_experiments.new_beta_varix_experiments import (
    build_varix_config, prepare_fixed_data_splits, save_split_manifest, save_run_loss_history)
from beta_vae_experiments.data_pipeline import build_data_config, save_epoch_snapshots
from beta_vae_experiments.prototype_experiments.preservation_annealing import _self_first_neighbours, seek_preservation_target, \
    PreservationAnnealingConfig, PreservationStage
from beta_vae_experiments.metrics.information_theoretic import compute_conditional_kl_matrix
from beta_vae_experiments.metrics.latent_usage import PosteriorCollapseConfig
from beta_vae_experiments.metrics.evaluation.downstream import ClassAUCEvaluator, snapshot_tasks
from beta_vae_experiments.prototype_experiments.live_dashboard import LiveDashboard

OUTPUT = PROJECT_ROOT / "beta_vae_experiments" / "outputs" / "trustworthiness_annealer"
RUNS = OUTPUT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
COLLAPSE = PosteriorCollapseConfig()  # ε = 0,01 Nats, δ = 1 %
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "legend.frameon": False})

## 1. Der Annealer

Zwei Klassen, beide über die öffentlichen AUTOENCODIX-Schnittstellen:

* `MaximinAnnealer(AnnealingScheduler)` erbt den AUTOENCODIX-Annealer und überschreibt nur `get_weight`. Er liefert das
  aktuelle Gewicht w ∈ [0, 1]; `VarixLoss` rechnet wie immer `β_eff = β_config · w`. Mit `native=True` gibt er unverändert
  die AUTOENCODIX-Kurve (z. B. `logistic-mid`) zurück und misst nur mit: so laufen die Vergleichsläufe ohne Regelung
  durch denselben Code.
* `annealed_varix_loss(...)` erzeugt eine `VarixLoss`-Unterklasse, die in der Validierung (ohne Gradient) X und μ des festen
  Subsets einsammelt, am Epochenende misst und dem Annealer die Entscheidung überlässt. Der Loss selbst bleibt der von
  AUTOENCODIX.

**Regel** (pro Messung, alle `every` Epochen nach dem Warm-up): Signal s = min_i m̃_i.
s > 1 → β um den Faktor e^Schritt erhöhen; s < −1 → senken; |s| ≤ 1 → halten. Bei jedem Richtungswechsel wird der
log-Schritt halbiert (minimal 0,1 %). Mit einer einzigen Metrik ist das genau `seek_preservation_target` der bisherigen
Läufe (Test unten). Ein ungültiger Messwert zählt als s = −∞ (β runter), nie als Erfolg.

Das Log (`controller_log.csv`) enthält pro Epoche den Zustand, *mit dem die Epoche trainiert wurde*; die Entscheidung aus
einer Messung wirkt ab der nächsten Epoche. Zusätzlich werden pro Epoche Rate und ε-δ-Collapse auf allen
Validierungspatienten mitgeschrieben (nur Diagnose).

In [2]:
@dataclass(frozen=True)
class Constraint:
    """Nebenbedingung m(X, μ) ≥ target für eine höher-ist-besser-Metrik."""
    name: str
    measure: Callable[[np.ndarray, np.ndarray], float]
    target: float
    tolerance: float = 0.01


def trustworthiness_constraint(target, k=20, tolerance=0.01):
    return Constraint(f"trustworthiness_k{k}", lambda x, mu: float(trustworthiness(x, mu, n_neighbors=k)), target, tolerance)


class MaximinAnnealer(AnnealingScheduler):
    """β so groß wie möglich, solange die schlechteste normierte Metrik im Sollband oder darüber liegt."""

    def __init__(self, constraints=(), *, warmup_epochs=100, every=5, growth=1.25, w_min=1e-4,
                 min_change=1e-3, native=False):
        super().__init__()
        self.constraints = tuple(constraints)
        if not self.constraints:
            raise ValueError("Mindestens eine Nebenbedingung (auch native Läufe messen mit).")
        self.warmup_epochs, self.every, self.w_min, self.native = warmup_epochs, every, w_min, native
        self.min_step = float(np.log1p(min_change))
        self.w, self.step, self.direction = 0.0, float(np.log(growth)), 0
        self.state, self.reason = ("native", "") if native else ("warmup", "")

    def get_weight(self, *, epoch_current, total_epoch, func="logistic-mid"):
        if self.native:
            return super().get_weight(epoch_current=epoch_current, total_epoch=total_epoch, func=func)
        return self.w

    def measures_at(self, epoch, total_epochs):
        """epoch ist 0-basiert wie in AUTOENCODIX; gemessen wird nach dem Training dieser Epoche."""
        done = epoch + 1
        return done == self.warmup_epochs or done == total_epochs or (
            done > self.warmup_epochs and (done - self.warmup_epochs) % self.every == 0)

    def evaluate(self, x, mu):
        values, margins = {}, {}
        for constraint in self.constraints:
            try:
                value = constraint.measure(x, mu) if len(np.unique(mu, axis=0)) > 1 else np.nan
            except (ValueError, RuntimeError, ZeroDivisionError):
                value = np.nan
            values[constraint.name] = value
            margins[constraint.name] = (value - constraint.target) / constraint.tolerance if np.isfinite(value) else -np.inf
        return values, margins

    def update(self, margins):
        """Ein Dual-Schritt auf log β. Gibt das Signal s = min_i m̃_i zurück."""
        signal = min(margins.values())
        if self.native:
            return signal
        if abs(signal) <= 1 + 1e-12:
            self.state, self.reason = "hold", "target_band"
            return signal
        direction = 1 if signal > 0 else -1
        if self.direction and direction != self.direction:
            self.step = max(self.step / 2, self.min_step)
        weight = self.w_min if self.w == 0 and direction > 0 else self.w * np.exp(direction * self.step)
        weight = float(np.clip(weight, 0.0, 1.0))
        if direction < 0 and weight < self.w_min:
            weight = 0.0
        bounded = weight == self.w
        self.reason = ("weight_cap" if direction > 0 else "zero_weight_bound") if bounded else (
            "above_target" if direction > 0 else "below_target")
        self.state = "hold" if bounded else "seeking"
        self.w, self.direction = weight, direction
        return signal


def annealed_varix_loss(annealer, folder, valid_ids, positions, on_measure=None, weights=None):
    """VarixLoss-Unterklasse, die für den Annealer misst. Zustand liegt im Closure, weil AUTOENCODIX den Loss
    für Training und Vorhersage getrennt instanziiert (`loss_type(config=...)`).

    `weights=dict(dense=(Präfixe...), full_every=N)` schneidet Gewichte mit: die Schichten am Flaschenhals (Präfixe)
    nach jeder Epoche, den ganzen `state_dict` alle N Epochen. AUTOENCODIX' eigene `model_checkpoints` taugen dafür
    nicht: `_store_checkpoint` legt `state_dict()` ohne Kopie ab, das sind Referenzen auf die laufenden Parameter."""
    folder = Path(folder)
    n_valid = len(valid_ids)
    shared = dict(reference=None, rows=[], logged=set())
    recorder = None if weights is None else dict(model=None, epochs=[], dense={})

    def find_model():
        # Der Loss bekommt von AUTOENCODIX nur die config; das Modell wird einmal im Aufrufer (Trainer) gesucht und nur gelesen.
        frame = sys._getframe(1)
        while frame is not None:
            owner = frame.f_locals.get("self")
            if hasattr(owner, "_model") and hasattr(owner, "_optimizer"):
                return owner._model
            frame = frame.f_back
        raise RuntimeError("Trainer mit _model nicht im Aufrufstapel gefunden.")

    def record_weights(epoch):
        state = {name.replace("_forward_module.", "").replace("_orig_mod.", ""): value.detach().cpu().numpy().copy()
                 for name, value in recorder["model"].state_dict().items()}
        recorder["epochs"].append(epoch + 1)
        for name, value in state.items():
            if name.startswith(tuple(weights["dense"])):
                recorder["dense"].setdefault(name, []).append(value)
        target = folder / "weights"
        target.mkdir(exist_ok=True)
        if (epoch + 1) % weights["full_every"] == 0 or epoch == annealer_total[0] - 1:
            np.savez_compressed(target / f"full_{epoch + 1:05d}.npz", **state)
            np.savez_compressed(target / "dense.npz", epoch=np.asarray(recorder["epochs"]),
                                **{name: np.stack(values) for name, values in recorder["dense"].items()})

    annealer_total = [None]

    class AnnealedVarixLoss(VarixLoss):
        def __init__(self, config, annealing_scheduler=None):
            if annealing_scheduler is not None:
                raise ValueError("Der Annealer ist bereits gebunden.")
            if config.anneal_function == "no-annealing" or config.default_vae_loss != "kl" or config.loss_reduction != "mean":
                raise ValueError("Braucht KL, mean-Reduktion und eine aktive anneal_function (sonst wird get_weight ignoriert).")
            super().__init__(config, annealing_scheduler=annealer)
            self.epoch, self.offset = None, 0
            self.inputs, self.means, self.logvars = [], [], []

        def flush(self):
            if self.epoch is None or self.epoch in shared["logged"]:
                return
            if self.offset != n_valid:
                raise ValueError("Validierungs-Loader hat nicht die erwartete geordnete Kohorte geliefert.")
            mu, logvar = np.concatenate(self.means), np.concatenate(self.logvars)
            if shared["reference"] is None:
                shared["reference"] = np.concatenate(self.inputs)
                np.savez_compressed(folder / "controller_validation_reference.npz", x=shared["reference"],
                                    sample_ids=np.asarray(valid_ids, dtype=str)[positions], validation_positions=positions)
            total = self.config.epochs
            weight = annealer.get_weight(epoch_current=self.epoch, total_epoch=total, func=self.config.anneal_function)
            kl = compute_conditional_kl_matrix(mu, logvar)
            row = dict(epoch=self.epoch + 1, w=weight, effective_beta=self.config.beta * weight,
                       beta_over_d=self.config.beta * weight / self.config.latent_dim,
                       state=annealer.state, reason=annealer.reason,
                       rate=float(kl.sum(1).mean()),
                       collapsed_dimensions=int(np.sum(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)),
                       zero_patients=float((mu == 0).all(1).mean()),
                       metric=",".join(c.name for c in annealer.constraints), metric_value=np.nan, signal=np.nan,
                       target=annealer.constraints[0].target, tolerance=annealer.constraints[0].tolerance,
                       seed=self.config.global_seed, latent_dim=self.config.latent_dim)
            measured = annealer.measures_at(self.epoch, total)
            if measured:
                subset = mu[positions].astype(np.float64)
                values, margins = annealer.evaluate(shared["reference"], subset)
                binding = min(margins, key=margins.get)
                constraint = next(c for c in annealer.constraints if c.name == binding)
                row.update({f"value_{name}": value for name, value in values.items()},
                           metric_value=values[binding], target=constraint.target, tolerance=constraint.tolerance,
                           signal=margins[binding], distinct_mu_share=len(np.unique(subset, axis=0)) / len(subset))
            shared["rows"].append(row)
            shared["logged"].add(self.epoch)
            log = pd.DataFrame(shared["rows"])
            log.to_csv(folder / "controller_log.csv", index=False)
            # Entscheidung erst nach dem Warm-up und nicht nach der letzten trainierten Epoche.
            if measured and self.epoch + 1 >= annealer.warmup_epochs and self.epoch < total - 1:
                annealer.update(margins)
                if on_measure is not None:
                    on_measure(log)
            if recorder is not None and recorder["model"] is not None:
                record_weights(self.epoch)
            self.offset, self.inputs, self.means, self.logvars = 0, [], [], []

        def forward(self, model_output, targets, epoch=None, **kwargs):
            if recorder is not None and recorder["model"] is None and torch.is_grad_enabled():
                recorder["model"], annealer_total[0] = find_model(), self.config.epochs
            if epoch is not None and torch.is_grad_enabled() and epoch != self.epoch:
                self.flush()
                self.epoch = int(epoch)
            if epoch is not None and not torch.is_grad_enabled():
                local = positions[(positions >= self.offset) & (positions < self.offset + len(targets))] - self.offset
                if shared["reference"] is None and len(local):
                    self.inputs.append(targets[local].detach().cpu().numpy())
                self.means.append(model_output.latent_mean.detach().cpu().numpy())
                self.logvars.append(model_output.latent_logvar.detach().cpu().numpy())
                self.offset += len(targets)
            result = super().forward(model_output=model_output, targets=targets, epoch=epoch, **kwargs)
            if epoch == self.config.epochs - 1 and not torch.is_grad_enabled() and self.offset == n_valid:
                self.flush()
            return result

    return AnnealedVarixLoss

**Test:** Mit einer Nebenbedingung trifft `MaximinAnnealer.update` dieselben Entscheidungen wie der bisherige Controller
`seek_preservation_target` (gleiche Messfolge, gleiche Gewichte). Die Scores sind künstlich und decken Erhöhen, Halten,
Senken und Richtungswechsel ab.

In [3]:
rng = np.random.default_rng(0)
scores = np.concatenate([np.linspace(.97, .9, 30), .9 + .03 * rng.standard_normal(60), np.linspace(.88, .8, 10)])
new = MaximinAnnealer([Constraint("t", None, .9, .01)])
old_policy = PreservationAnnealingConfig(target=.9, rule="target", tolerance=.01, w_min=1e-4, growth=1.25)
old = PreservationStage(state="seeking")
for score in scores:
    new.update({"t": (score - .9) / .01})
    old = seek_preservation_target(old, score, .9, old_policy)
    assert np.isclose(new.w, old.w, rtol=1e-12, atol=0) and new.state == old.state, (score, new.w, old.w)
print(f"{len(scores)} Entscheidungen identisch; letztes w = {new.w:.3g}")

100 Entscheidungen identisch; letztes w = 0.0238


## 2. Daten und Versuchsplan

Gleiche Vorverarbeitung wie alle bisherigen TCGA-Läufe (AUTOENCODIX, MINMAX-Skalierung, k_filter = 2000, Split-Seed 2),
d = 8, 600 Epochen, Lernrate 5·10⁻⁴, Controller wie im Pilot: Warm-up 100 Epochen (β = 0), Messung alle 5 Epochen auf
800 festen Validierungspatienten, Startgewicht 10⁻⁴, Wachstum 1,25, β/d höchstens 0,015. Snapshots alle 20 Epochen.

| Block | Läufe | Wofür |
|---|---|---|
| A | Trustworthiness (k = 20), Ziel 0,95 / 0,925 / 0,90 / 0,875 / 0,85 / 0,80 / 0,75 / 0,70 | Fragen 1–5: Front Rate ↔ Preservation ↔ AUC |
| B | Ziel 0,90 (absolut) mit Controller-k = 10 / 40 / 200 | Frage 6 (k = 20 steckt in A). Bei k = 200 liegt T schon am Warm-up-Ende unter 0,90; der Controller erhöht β dann nie (β = 0-Lauf). Ein fairer k-Vergleich bräuchte relative Ziele. |
| C | ohne Regelung: AUTOENCODIX `logistic-mid` bis β/d = 1,5·10⁻³ bzw. 4·10⁻³ | Gegenprobe: gleicher β-Bereich ohne Rückkopplung |

Jeder Block mit den Seeds 17, 23 und 31; zuerst alle Läufe mit Seed 17, damit früh ein vollständiges Bild entsteht.
Fertige Läufe (`run.json` im Lauf-Ordner) werden übersprungen; die Auswertung unten nimmt, was fertig ist.

In [4]:
EPOCHS, LATENT_DIM, SNAPSHOT_EVERY = 600, 8, 20
BETA_OVER_D_MAX = 0.015
CONTROLLER = dict(warmup_epochs=100, every=5, growth=1.25, w_min=1e-4)
N_SUBSET, SUBSET_SEED = 800, 31
SEEDS = (17, 23, 31)
TARGETS = (0.95, 0.925, 0.90, 0.875, 0.85, 0.80, 0.75, 0.70)
CONTROLLER_KS = (10, 40, 200)
FIXED_BETA_OVER_D = (1.5e-3, 4e-3)


@dataclass(frozen=True)
class RunSpec:
    block: str
    seed: int
    target: float = np.nan
    k: int = 20
    fixed_beta_over_d: float = np.nan

    @property
    def name(self):
        if self.block == "C":
            return f"C_fixed_bd{self.fixed_beta_over_d:.1e}_s{self.seed}"
        return f"{self.block}_T{self.target:.3f}_k{self.k}_s{self.seed}"


grid = []
for seed in SEEDS:
    grid += [RunSpec("A", seed, target) for target in TARGETS]
    grid += [RunSpec("B", seed, 0.90, k) for k in CONTROLLER_KS]
    grid += [RunSpec("C", seed, fixed_beta_over_d=value) for value in FIXED_BETA_OVER_D]
plan = pd.DataFrame([dict(name=s.name, block=s.block, seed=s.seed, target=s.target, k=s.k,
                          fixed_beta_over_d=s.fixed_beta_over_d, fertig=(RUNS / s.name / "run.json").exists()) for s in grid])
print(f"{len(plan)} Läufe, davon fertig: {plan.fertig.sum()}")

dataset = dict(id="tcga", split_seed=2, options=dict(
    data_root="data/raw/TCGAPanCancerAutoencodix", rna_file="data_mrna_seq_v2_rsem_formatted.parquet",
    methylation_file="data_methylation_per_gene_formatted.parquet", clinical_file="data_clinical_formatted.parquet"))
dataset["options"]["data_root"] = str((PROJECT_ROOT / dataset["options"]["data_root"]).resolve())
TRAINING = dict(epochs=EPOCHS, anneal_function="logistic-mid", device="cuda", checkpoint_interval=SNAPSHOT_EVERY,
                snapshots=dict(enabled=True, every=SNAPSHOT_EVERY), downstream_targets=["CANCER_TYPE_ACRONYM", "SEX"])
RUN_DEFAULTS = dict(latent_dim=LATENT_DIM, loss_reduction="mean", learning_rate=5e-4, loss="kl")

data_config = build_data_config(PROJECT_ROOT, "tcga", dataset["options"])
splits = prepare_fixed_data_splits(None, build_varix_config(
    data_config, "tcga", TRAINING, dict(RUN_DEFAULTS, beta=0.0), global_seed=dataset["split_seed"]))
save_split_manifest(splits, OUTPUT)
VALID_IDS = np.asarray(splits.valid.sample_ids, dtype=str)
POSITIONS = np.sort(np.random.default_rng(SUBSET_SEED).choice(len(VALID_IDS), N_SUBSET, replace=False))
print(f"Train {len(splits.train.sample_ids)}, Validierung {len(VALID_IDS)}, Controller-Subset {len(POSITIONS)}")

39 Läufe, davon fertig: 39
reading parquet: /home/nickok/Projects/MasterThesis/autoencodix_package/data/raw/TCGAPanCancerAutoencodix/data_mrna_seq_v2_rsem_formatted.parquet
reading parquet: /home/nickok/Projects/MasterThesis/autoencodix_package/data/raw/TCGAPanCancerAutoencodix/data_methylation_per_gene_formatted.parquet
reading parquet: /home/nickok/Projects/MasterThesis/autoencodix_package/data/raw/TCGAPanCancerAutoencodix/data_clinical_formatted.parquet
anno key: paired
Train 7009, Validierung 1001, Controller-Subset 800


## 3. Training

Läuft im Notebook-Kernel mit dem nativen AUTOENCODIX-Trainer (`acx.Varix(..., loss_type=...)`). Die Live-Anzeige zeigt für den
laufenden Lauf β/d, das Steuersignal mit Sollband und die Rate mit ε-δ-Collapse; sie wird nach jeder Messung neu gezeichnet.
Nach jedem Lauf werden die Snapshots gespeichert und das Live-Dashboard (`live_dashboard.py`) rechnet für jeden Snapshot den
labelfreien Cluster-Overlap (`live_metrics.csv`, `live_summary.md` im Lauf-Ordner).
Ein Lauf mit 600 Epochen dauert etwa 16 Minuten, dazu etwa 2 Minuten Auswertung.

In [5]:
RUN_TRAINING = True

live_handle = display(Markdown("_Noch keine Messung._"), display_id=True)


def live_plot(spec):
    def draw(log):
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", color="black")
        axes[0].set(yscale="log", title="β/d", xlabel="Epoche")
        measured = log.dropna(subset=["metric_value"])
        axes[1].plot(measured.epoch, measured.metric_value, "o-", ms=2.5)
        if not spec.block == "C":
            axes[1].axhspan(spec.target - .01, spec.target + .01, alpha=.15)
        axes[1].set(title=f"Trustworthiness (k = {spec.k})", xlabel="Epoche")
        axes[2].plot(log.epoch, log.rate, lw=1.2)
        collapsed = log[log.collapsed_dimensions > 0]
        axes[2].plot(collapsed.epoch, collapsed.rate, "x", color="black", ms=4, label="≥ 1 kollabierte Dim.")
        axes[2].set(yscale="log", title="Rate (alle Validierungspatienten)", xlabel="Epoche")
        if len(collapsed):
            axes[2].legend(fontsize=8)
        fig.suptitle(spec.name)
        fig.tight_layout()
        live_handle.update(fig)
        plt.close(fig)
    return draw


def train(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    if spec.block == "C":
        beta_max = spec.fixed_beta_over_d
        annealer = MaximinAnnealer([trustworthiness_constraint(0.9, spec.k)], native=True, **CONTROLLER)
    else:
        beta_max = BETA_OVER_D_MAX
        annealer = MaximinAnnealer([trustworthiness_constraint(spec.target, spec.k)], **CONTROLLER)
    config = build_varix_config(data_config, "tcga", TRAINING, dict(RUN_DEFAULTS, beta=beta_max * LATENT_DIM),
                                global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, beta_over_d_max=beta_max, controller=CONTROLLER, n_subset=N_SUBSET, subset_seed=SUBSET_SEED,
        epochs=EPOCHS, anneal_function=config.anneal_function, native=annealer.native), indent=2, default=float) + "\n")
    loss_type = annealed_varix_loss(annealer, folder, VALID_IDS, POSITIONS, on_measure=live_plot(spec))
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=SNAPSHOT_EVERY, downstream=False)
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    LiveDashboard(folder).update_metrics()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, train_seconds=trained,
                                                     total_seconds=time.time() - started), default=float) + "\n")


if RUN_TRAINING:
    todo = [spec for spec in grid if not (RUNS / spec.name / "run.json").exists()]
    for seed in SEEDS:  # erst alle Läufe eines Seeds
        for spec in [s for s in todo if s.seed == seed]:
            print(f"{time.strftime('%H:%M')} Lauf {spec.name}", flush=True)
            train(spec)
print("fertig:", sum((RUNS / s.name / "run.json").exists() for s in grid), "von", len(grid))

_Noch keine Messung._

fertig: 39 von 39


## 4. Auswertung je Lauf

Pro Lauf und Snapshot (alle 40 Epochen, dazu Ende Warm-up und Laufende) auf den **gespeicherten Snapshots**:

* Rate (mittlere KL, Nats), kollabierte Dimensionen nach ε-δ, Patienten mit μ = 0 in allen Dimensionen
* Trustworthiness (k = 20) auf dem Controller-Subset, X → μ
* AUC auf z (logistische Regression, Train → Validation): Krebsart (33) und Geschlecht **nur innerhalb gemischter
  Krebsarten** (`snapshot_tasks`, `sex_mixed_cancers`), weil die Krebsart Geschlecht sonst schon zu AUC 0,83 vorhersagt
* Labelfreier Cluster-Overlap aus `live_metrics.csv`: Anteil der 20 nächsten Nachbarn im eigenen X-Cluster (μ und z), ARI

Am Laufende zusätzlich der k-Sweep: Trustworthiness und R_NX(k) = (n−1)·LCMC/(n−1−k) für k = 5 … 200 auf μ und z.
Ergebnisse werden pro Lauf in `analysis_trajectory.csv` und `analysis_final.json` gespeichert und nur einmal gerechnet.

In [5]:
K_SWEEP = (5, 10, 20, 40, 80, 200)


def load_snapshot(folder, epoch, split):
    with np.load(folder / "epoch_snapshots" / f"epoch_{epoch:05d}_{split}.npz") as saved:
        arrays = {key: saved[key] for key in saved.files} | dict(sample_ids=saved["sample_ids"].astype(str))
    # AUTOENCODIX drops mu when its buffer sums to 0 (_general_trainer.py, maybe_add); with the Varix clamp
    # mu >= 0, so a missing mu means mu == 0 for every patient (total collapse), not a lost array.
    arrays.setdefault("mu", np.zeros_like(arrays["logvar"]))
    return arrays


def labels_of(folder, split):
    labels = pd.read_parquet(folder / "epoch_snapshots" / f"labels_{split}.parquet")
    labels.index = labels.index.astype(str)
    return labels


def auc_on_z(folder, epoch):
    """Krebs-AUC und Geschlechts-AUC innerhalb gemischter Krebsarten auf z (Train → Validation)."""
    snaps = {split: load_snapshot(folder, epoch, split) for split in ("train", "valid")}
    labels = {split: labels_of(folder, split).loc[snaps[split]["sample_ids"]] for split in snaps}
    tasks = {task["name"]: task for task in snapshot_tasks(labels)}
    scores = {}
    for name, task in (("auc_cancer", tasks["cancer_type"]), ("auc_sex_mixed", tasks["sex_mixed_cancers"])):
        frames = {s: pd.DataFrame(snaps[s]["z"], index=snaps[s]["sample_ids"])[task["keep"][s]] for s in snaps}
        partition = pd.concat([pd.DataFrame(dict(SAMPLE_ID=f.index, SPLIT=s)) for s, f in frames.items()], ignore_index=True)
        annotation = pd.concat([task["y"][s][task["keep"][s]].rename(task["target"]) for s in snaps]).to_frame()
        result = ClassAUCEvaluator()._single_ml_presplit(partition, pd.concat(frames.values()), annotation, task["target"],
                                                         LogisticRegression(max_iter=1000), "roc_auc_ovo",
                                                         "classification", top_k_classes=None)
        scores[name] = float(result.loc[result.score_split.eq("valid"), "value"].iloc[0])
    return scores


def r_nx(x, embedding, k):
    x_neighbours, _ = _self_first_neighbours(x, k)
    neighbours, _ = _self_first_neighbours(embedding, k)
    lcmc = local_continuity_meta_criteria.measure(x, embedding, k=k, knn_info=(x_neighbours, neighbours))["lcmc"]
    return (len(x) - 1) * lcmc / (len(x) - 1 - k)


def analyse_run(spec):
    folder = RUNS / spec.name
    trajectory_path, final_path = folder / "analysis_trajectory.csv", folder / "analysis_final.json"
    if trajectory_path.exists() and final_path.exists():
        return pd.read_csv(trajectory_path), json.loads(final_path.read_text())
    controller = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    live = pd.read_csv(folder / "live_metrics.csv").set_index("epoch")
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    epochs = sorted({CONTROLLER["warmup_epochs"], EPOCHS} | set(range(40, EPOCHS + 1, 40)))
    rows = []
    for epoch in epochs:
        valid = load_snapshot(folder, epoch, "valid")
        positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
        mu = valid["mu"][positions].astype(np.float64)
        kl = compute_conditional_kl_matrix(valid["mu"], valid["logvar"])
        rows.append(dict(epoch=epoch, beta_over_d=controller.loc[epoch, "beta_over_d"], rate=float(kl.sum(1).mean()),
                         collapsed_dimensions=int(np.sum(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)),
                         zero_patients=float((valid["mu"] == 0).all(1).mean()),
                         distinct_mu_share=len(np.unique(mu, axis=0)) / len(mu),
                         trustworthiness=float(trustworthiness(x, mu, n_neighbors=20)) if len(np.unique(mu, axis=0)) > 1 else np.nan,
                         **auc_on_z(folder, epoch),
                         same_cluster_mu=live.loc[epoch, "same_cluster_neighbours_latent"],
                         same_cluster_z=live.loc[epoch, "same_cluster_neighbours_z"],
                         same_cluster_x=live.loc[epoch, "same_cluster_neighbours_x"], chance=live.loc[epoch, "chance"],
                         recovery_ari=live.loc[epoch, "recovery_ari"], n_clusters_latent=live.loc[epoch, "n_clusters_latent"]))
    trajectory = pd.DataFrame(rows)
    valid = load_snapshot(folder, EPOCHS, "valid")
    positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
    final = dict(name=spec.name)
    for space in ("mu", "z"):
        values = valid[space][positions].astype(np.float64)
        for k in K_SWEEP:
            degenerate = len(np.unique(values, axis=0)) <= k
            final[f"t_{space}_k{k}"] = np.nan if degenerate else float(trustworthiness(x, values, n_neighbors=k))
            final[f"rnx_{space}_k{k}"] = np.nan if degenerate else float(r_nx(x, values, k))
    trajectory.to_csv(trajectory_path, index=False)
    final_path.write_text(json.dumps(final, indent=1) + "\n")
    return trajectory, final


done = [spec for spec in grid if (RUNS / spec.name / "run.json").exists()]
trajectories, finals = [], []
for spec in done:
    trajectory, final = analyse_run(spec)
    trajectories.append(trajectory.assign(**{k: v for k, v in spec.__dict__.items()}, name=spec.name))
    finals.append(final)
trajectories = pd.concat(trajectories, ignore_index=True)
warmup = trajectories[trajectories.epoch.eq(CONTROLLER["warmup_epochs"])].set_index("name")
last = trajectories[trajectories.epoch.eq(EPOCHS)].set_index("name")
ever_collapsed = trajectories[trajectories.epoch > CONTROLLER["warmup_epochs"]].groupby("name").collapsed_dimensions.max()
results = last.join(pd.DataFrame(finals).set_index("name")).assign(
    rate_warmup=warmup.rate, t_warmup=warmup.trustworthiness, auc_cancer_warmup=warmup.auc_cancer,
    auc_sex_mixed_warmup=warmup.auc_sex_mixed, max_collapsed_dimensions=ever_collapsed)
results["rate_saving"] = 1 - results.rate / results.rate_warmup
results.to_csv(OUTPUT / "results.csv")
print(f"{len(results)} Läufe ausgewertet")

39 Läufe ausgewertet


## 5. Frage 1 und 2: Collapse und eingesparte Rate

Pro Sollwert über die Seeds: End-β/d, Rate am Ende des Warm-ups (β = 0) und am Laufende, eingesparte Rate
(1 − Rate_Ende / Rate_Warm-up; gegen β = 0, wo μ ungebremst wächst und die Rate künstlich groß ist, deshalb nur als
Größenordnung zu lesen; aussagekräftiger ist die absolute Rate), Anzahl Seeds mit mindestens einer kollabierten Dimension (ε-δ, irgendwann nach dem
Warm-up), Patienten bei μ = 0 am Ende und Trustworthiness am Ende. Block C ohne Regelung zum Vergleich.

In [6]:
def setting(row):
    if row.block == "C":
        return f"fest β/d = {row.fixed_beta_over_d:.1e}"
    return f"Ziel {row.target:.3f}, k = {int(row.k)}"


results["setting"] = results.apply(setting, axis=1)
order = results.sort_values(["block", "target", "k", "fixed_beta_over_d"], ascending=[True, False, True, True]).setting.unique()
summary = results.groupby("setting").agg(
    seeds=("seed", "size"), beta_over_d=("beta_over_d", "median"), rate_warmup=("rate_warmup", "median"),
    rate=("rate", "median"), rate_min=("rate", "min"), rate_max=("rate", "max"),
    rate_saving=("rate_saving", "median"), seeds_with_collapse=("max_collapsed_dimensions", lambda s: int((s > 0).sum())),
    collapsed_end=("collapsed_dimensions", "max"), zero_patients=("zero_patients", "max"),
    trustworthiness=("trustworthiness", "median"), t_warmup=("t_warmup", "median")).reindex(order)
display(summary.style.format({"beta_over_d": "{:.2e}", "rate_warmup": "{:.2f}", "rate": "{:.3f}", "rate_min": "{:.3f}",
                              "rate_max": "{:.3f}", "rate_saving": "{:.1%}", "zero_patients": "{:.1%}",
                              "trustworthiness": "{:.3f}", "t_warmup": "{:.3f}"})
        .set_caption("Median über Seeds (Rate-Spanne min–max); Collapse = Seeds mit ≥ 1 kollabierter Dimension nach dem Warm-up"))
summary.to_csv(OUTPUT / "summary_by_setting.csv")

,seeds,beta_over_d,rate_warmup,rate,rate_min,rate_max,rate_saving,seeds_with_collapse,collapsed_end,zero_patients,trustworthiness,t_warmup
setting,,,,,,,,,,,,
"Ziel 0.950, k = 20",3,2.03e-04,3197.03,14.080,13.715,21.112,99.3%,0,0,2.3%,0.951,0.960
"Ziel 0.925, k = 20",3,6.94e-04,3197.03,5.639,4.504,5.875,99.8%,0,0,6.3%,0.923,0.960
"Ziel 0.900, k = 20",3,1.08e-03,3197.03,3.162,2.953,3.451,99.9%,0,0,6.9%,0.901,0.960
"Ziel 0.875, k = 20",3,1.21e-03,3197.03,2.517,2.346,3.213,99.9%,0,0,6.8%,0.877,0.960
"Ziel 0.850, k = 20",3,1.35e-03,3197.03,1.973,1.460,2.660,100.0%,1,2,7.7%,0.848,0.960
"Ziel 0.800, k = 20",3,2.09e-03,3197.03,1.200,1.148,1.258,100.0%,2,4,13.2%,0.799,0.960
"Ziel 0.750, k = 20",3,2.45e-03,3197.03,0.804,0.756,1.258,100.0%,3,5,40.2%,0.758,0.960
"Ziel 0.700, k = 20",3,2.06e-03,3197.03,0.661,0.371,0.747,100.0%,3,6,57.3%,0.695,0.960
"Ziel 0.900, k = 10",3,9.69e-04,3197.03,3.381,3.155,3.469,99.9%,0,0,6.2%,0.897,0.960


## 6. Frage 3 und 4: Preservation und AUC gegen die Rate

Jeder Punkt ist ein Lauf am Ende (600 Epochen); die Linien sind die Verläufe des Seed-17-Laufs je Sollwert über die
Snapshots (von rechts, hohe Rate am Warm-up-Ende, nach links). Graue Kreuze: Block C ohne Regelung. Die x-Achse ist
logarithmisch; die senkrechte Linie bei 0,5 Nats markiert SNR = 1 (E‖μ‖² = σ²), darunter beginnt in den bisherigen
Läufen der massive Collapse.

In [7]:
A = results[results.block.eq("A")]
C = results[results.block.eq("C")]
colors = dict(zip(TARGETS, plt.cm.viridis(np.linspace(0, .9, len(TARGETS)))))
panels = [("trustworthiness", "Trustworthiness (k = 20, μ)"), ("rnx_z_k10", "R_NX k = 10 (z)"),
          ("rnx_z_k200", "R_NX k = 200 (z)"), ("auc_cancer", "Krebs-AUC (z)"),
          ("auc_sex_mixed", "Geschlechts-AUC gemischte Krebsarten (z)"), ("same_cluster_z", "Nachbarn im eigenen X-Cluster (z)")]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for axis, (column, title) in zip(axes.ravel(), panels):
    for target in TARGETS:
        group = A[A.target.eq(target)]
        axis.scatter(group.rate, group[column], color=colors[target], s=40, label=f"Ziel {target:g}", zorder=3)
        path = trajectories[(trajectories.block.eq("A")) & trajectories.target.eq(target) & trajectories.seed.eq(17)
                            & (trajectories.epoch >= CONTROLLER["warmup_epochs"])]
        if column in path:
            axis.plot(path.rate, path[column], "-", color=colors[target], lw=.8, alpha=.6)
    axis.scatter(C.rate, C[column], marker="x", color="gray", s=50, label="ohne Regelung (C)", zorder=3)
    axis.axvline(.5, color="red", ls=":", lw=.8)
    axis.set(xscale="log", xlabel="Rate am Laufende [Nats]", title=title)
axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT / "front_rate_preservation_auc.png", dpi=150)
plt.show()

### 6.1 Preservation über k

Wie stark fällt die Erhaltung pro Nachbarschaftsgröße? R_NX und Trustworthiness am Laufende gegen den Sollwert, je k eine
Linie (Median über Seeds), auf μ und z. Links (hoher Sollwert) ist wenig Druck, rechts viel.

In [8]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
for axis, (prefix, title) in zip(axes, [("t_mu", "Trustworthiness μ"), ("t_z", "Trustworthiness z"),
                                        ("rnx_mu", "R_NX μ"), ("rnx_z", "R_NX z")]):
    medians = A.groupby("target")[[f"{prefix}_k{k}" for k in K_SWEEP]].median()
    for k, color in zip(K_SWEEP, plt.cm.plasma(np.linspace(0, .85, len(K_SWEEP)))):
        axis.plot(medians.index, medians[f"{prefix}_k{k}"], "o-", ms=3, color=color, label=f"k = {k}")
    axis.set(title=title, xlabel="Trustworthiness-Sollwert (k = 20)")
    axis.invert_xaxis()
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "preservation_over_k.png", dpi=150)
plt.show()
k_table = A.groupby("target")[[f"rnx_z_k{k}" for k in K_SWEEP] + ["rate"]].median().sort_index(ascending=False)
display(k_table.style.format("{:.3f}").set_caption("R_NX auf z am Laufende (Median über Seeds) und Rate"))

,rnx_z_k5,rnx_z_k10,rnx_z_k20,rnx_z_k40,rnx_z_k80,rnx_z_k200,rate
target,,,,,,,
0.950000,0.129,0.188,0.253,0.309,0.321,0.359,14.080
0.925000,0.058,0.091,0.137,0.176,0.195,0.243,5.639
0.900000,0.031,0.056,0.088,0.117,0.134,0.181,3.162
0.875000,0.024,0.042,0.067,0.093,0.116,0.153,2.517
0.850000,0.018,0.027,0.045,0.068,0.090,0.121,1.973
0.800000,0.011,0.019,0.028,0.041,0.053,0.074,1.200
0.750000,0.007,0.012,0.021,0.033,0.038,0.058,0.804
0.700000,0.005,0.008,0.015,0.024,0.025,0.029,0.661


### 6.2 Signal, Rauschen und Informationsschranke gegen Trustworthiness

**Signal und Rauschen pro Dimension d** (über die Validierungspatienten eines Snapshots):

* Signal = Var_x(μ_d(x)): wie verschieden der Encoder die Patienten platziert. Das ist die Statistik
  A_u = Cov_x(E_{u∼q(u|x)}[u]) der „active units“ (Burda et al. 2016, Abschnitt 5.2; dort aktiv bei A_u > 10⁻²).
* Rauschen = E_x[σ_d²(x)]: die Breite der Posteriors, also die Streuung von z = μ + σ·ε um μ.
* SNR_d = Signal / Rauschen.

Active Units nach Burda (Zähler allein, Schwelle 10⁻²) und ε-δ-Collapse sind im Projekt schon vorhanden
(`metrics/latent_usage.py::compute_active_units`, Vergleich beider Kriterien in `trustworthiness_09.ipynb`, Abschnitt 2.5).
Neu ist hier die Normierung auf σ², die Zerlegung der Rate und der Bezug zu Trustworthiness auf μ und z.

**ε-δ-Collapse** ist die Definition von Lucas et al. (2019, Abschnitt 5): Dimension i ist (ε, δ)-kollabiert, wenn
P_x[KL(q(z_i|x) ‖ p(z_i)) < ε] ≥ 1 − δ; dort wie hier δ = 0,01 (wir: ε = 0,01 Nats).

**Varix-spezifisch (Code und Messung):** `autoencodix/modeling/_varix_architecture.py` klemmt `logvar` auf [0,01; 20] und setzt
μ < 10⁻⁷ auf 0. `torch.clamp` hat unterhalb der Grenze den Gradienten 0. Gemessen (unten): Sobald β/d über etwa 10⁻³
liegt, sitzt logvar in jedem Snapshot exakt bei 0,01, also σ² = e^0,01 ≈ 1,01 für alle Patienten und Dimensionen; das
Rauschen ist dann nicht gelernt. Abweichungen gibt es nur bei β = 0 und kurz nach dem Warm-up bei sehr kleinem β
(β/d ≤ 6·10⁻⁴, Epoche 120–240, Seeds 23 und 31). Die Zerlegung unten verwendet deshalb die allgemeine Form mit
E[log σ²]; sie gilt in allen Snapshots.

**Herleitung der Informationsschranke** (eigene Rechnung aus zwei Standardergebnissen):

1. Rate pro Dimension, Gauß-KL zum Prior N(0, 1): Rate_d = ½(E[μ_d²] + σ² − log σ² − 1).
2. Zerlegung nach Hoffman & Johnson (2016, Gl. 13–16): Rate_d = I(n; z_d) + KL(q(z_d) ‖ p(z_d)), mit n dem Index des
   Patienten (gleichverteilt über die N Patienten) und q(z_d) dem aggregierten Posterior; I(n; z_d) ≤ log N.
3. q(z_d) hat Mittelwert m = Mittel(μ_d) und Varianz V = Var(μ_d) + σ² (Gesetz der totalen Varianz). Bei gegebener
   Varianz hat die Normalverteilung die größte Entropie (Cover & Thomas 2006, Thm. 8.6.5), also
   KL(q(z_d) ‖ p) ≥ ½(m² + V − 1 − log V).
4. Abziehen, mit E[μ²] = Var(μ) + m²:

$$\text{Rate}_d = \tfrac12\log(1+\text{SNR}_d) + \tfrac12\,(m^2 + V - 1 - \log V), \qquad I(n; z_d) \le \tfrac12\log(1+\text{SNR}_d)$$

Die Identität ist exakt (bei konstantem σ²), die Ungleichung ist eine Obergrenze; Gleichheit nur, wenn q(z_d) normal ist.
Der zweite Summand ist Rate, die beweisbar keine Information über den Patienten trägt. Für alle Dimensionen gemeinsam gilt
mit demselben Satz I(n; z) ≤ ½ log det(I + Cov(μ)/σ²); die Summe der Einzelschranken ist nach der Hadamard-Ungleichung
(Cover & Thomas 2006, Kap. 17) größer oder gleich. Die Schranke ist hier eine Interpretationshilfe, keine Steuer- oder
Testgröße.

**Trustworthiness** (Venna & Kaski 2001; `sklearn.manifold.trustworthiness`) hängt nur von Nachbar-*Rängen* ab. Ränge
ändern sich nicht, wenn man alle μ mit einem Faktor c > 0 multipliziert; SNR hängt dagegen von dieser Skala ab. Deshalb
gibt es zwischen T(μ) und SNR **keine** analytische Beziehung (hergeleitet; geprüft unten mit c = 0,01). Für z = μ + σ·ε
hängen die Ränge nur von μ/σ ab, also von derselben Größe wie SNR: E[T(z)] ist bei gegebenem μ ohne Training berechenbar
(Monte Carlo über ε, fester Seed), eine geschlossene Form gibt es nicht, weil die Abstände eines Punktes zu seinen
Nachbarn korreliert sind.

Lucas et al. (2019) beweisen den Zusammenhang zwischen Beobachtungsrauschen und Collapse nur für den linearen VAE; für
tiefe VAEs (MNIST, CelebA) zeigen sie empirisch, dass ein großes festes Beobachtungsrauschen σ²_obs zu mehr kollabierten
Dimensionen führt (Tabelle 1). Bei einem Gauß-Decoder mit festem σ²_obs ist der Rekonstruktionsterm ‖x − x̂‖²/(2σ²_obs);
MSE + β·KL entspricht damit bis auf Konstanten und die Mittelungen über Merkmale und Dimensionen einem größeren σ²_obs bei
größerem β (eigene Umformung). Für unseren nichtlinearen Varix wird daraus **keine** Schwelle abgeleitet; die Richtung
(mehr β, mehr Collapse) ist mit ihren Ergebnissen vereinbar.

Unten: pro Lauf und Snapshot (Epoche 100 und alle 40 Epochen ab 120) SNR_d, Rate_d, beide Teile der Zerlegung, die
gemeinsame Schranke, T(μ) und E[T(z)] (Mittel über zwei Ziehungen) für k = 5 … 200 auf dem Controller-Subset. Die linearen
Anpassungen pro k sind **beschreibend** (keine analytische Vorhersage): T gegen die Schranke und gegen ihren Logarithmus,
nur Snapshots mit β > 0.

**Quellen**

* Burda, Grosse & Salakhutdinov (2016): Importance Weighted Autoencoders. ICLR. arXiv:1509.00519, Abschnitt 5.2.
* Hoffman & Johnson (2016): ELBO surgery: yet another way to carve up the variational evidence lower bound. NIPS Workshop
  on Advances in Approximate Bayesian Inference, Gl. 13–17.
* Cover & Thomas (2006): Elements of Information Theory, 2. Aufl., Wiley. Thm. 8.6.5 (Maximum-Entropie der
  Normalverteilung), Kap. 17 (Hadamard-Ungleichung).
* Venna & Kaski (2001): Neighborhood preservation in nonlinear projection methods: an experimental study. ICANN, LNCS 2130.
* Lucas, Tucker, Grosse & Norouzi (2019): Don't Blame the ELBO! A Linear VAE Perspective on Posterior Collapse. NeurIPS.
  arXiv:1911.02469. Abschnitt 5 ((ε, δ)-Collapse, Tabelle 1).

In [6]:
SNR_EPOCHS = [CONTROLLER["warmup_epochs"]] + list(range(120, EPOCHS + 1, 40))
Z_DRAWS = 2


def snr_table(spec):
    """Pro Snapshot: Zerlegung der Rate, Schranken und T auf μ und z. Zwischengespeichert in analysis_snr.csv."""
    folder = RUNS / spec.name
    cache = folder / "analysis_snr.csv"
    if cache.exists():
        return pd.read_csv(cache)
    controller = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    rows = []
    for epoch in SNR_EPOCHS:
        valid = load_snapshot(folder, epoch, "valid")
        mu, logvar = valid["mu"].astype(np.float64), valid["logvar"].astype(np.float64)
        variance = np.exp(logvar)
        noise, mean, signal = variance.mean(0), mu.mean(0), mu.var(0)
        snr = signal / noise
        total = noise + signal                                    # V = Var(μ) + E[σ²]
        rate_d = compute_conditional_kl_matrix(mu, logvar).mean(0)
        information_bound_d = 0.5 * (np.log(total) - np.log(variance).mean(0))   # = ½ log(1 + SNR) bei konstantem σ²
        no_information_d = 0.5 * (mean ** 2 + total - 1 - np.log(total))
        np.testing.assert_allclose(rate_d, information_bound_d + no_information_d, rtol=1e-9, atol=1e-9)
        scaled = mu / np.sqrt(noise)
        joint = 0.5 * np.linalg.slogdet(np.eye(mu.shape[1]) + np.cov(scaled.T, bias=True))[1]
        positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
        subset, sigma = mu[positions], np.sqrt(variance[positions])
        rng = np.random.default_rng(SUBSET_SEED)
        draws = [subset + sigma * rng.standard_normal(subset.shape) for _ in range(Z_DRAWS)]
        row = dict(epoch=epoch, beta_over_d=controller.loc[epoch, "beta_over_d"], rate=rate_d.sum(),
                   information_bound_sum=information_bound_d.sum(), information_bound_joint=joint,
                   no_information=no_information_d.sum(), logvar_max_deviation=np.abs(logvar - 0.01).max(),
                   collapsed_dimensions=int(np.sum(np.mean(compute_conditional_kl_matrix(mu, logvar) < COLLAPSE.primary_epsilon,
                                                           axis=0) >= 1 - COLLAPSE.delta)))
        for dim in range(mu.shape[1]):
            row |= {f"snr_{dim}": snr[dim], f"rate_{dim}": rate_d[dim], f"bound_{dim}": information_bound_d[dim]}
        distinct = len(np.unique(subset, axis=0))
        for k in K_SWEEP:
            row[f"t_mu_k{k}"] = trustworthiness(x, subset, n_neighbors=k) if distinct > k else np.nan
            row[f"t_z_k{k}"] = np.mean([trustworthiness(x, z, n_neighbors=k) for z in draws])
        if epoch == EPOCHS:  # Skaleninvarianz von T, eine Prüfung pro Lauf
            row["t_mu_k20_scaled_0p01"] = trustworthiness(x, 0.01 * subset, n_neighbors=20) if distinct > 20 else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(cache, index=False)
    return table


snr = pd.concat([snr_table(spec).assign(name=spec.name, block=spec.block, target=spec.target, k_controller=spec.k)
                 for spec in done], ignore_index=True)
final_snr = snr[snr.epoch.eq(EPOCHS)]
print(f"{snr.name.nunique()} Läufe, {len(snr)} Snapshots")
off_clamp = snr[snr.logvar_max_deviation > 1e-6]
print(f"logvar nicht exakt bei 0,01: {len(off_clamp)} von {len(snr)} Snapshots; davon mit β > 0: "
      f"{int((off_clamp.beta_over_d > 0).sum())}, größtes β/d dort {off_clamp.beta_over_d.max():.1e}, "
      f"Epochen {off_clamp[off_clamp.beta_over_d > 0].epoch.min()}–{off_clamp[off_clamp.beta_over_d > 0].epoch.max()}")
print("Zerlegung Rate = Schranke + Rest: in jedem Snapshot exakt geprüft (assert)")
print("T(0,01·μ) − T(μ) bei k = 20, Laufende, maximal:",
      f"{(final_snr.t_mu_k20_scaled_0p01 - final_snr.t_mu_k20).abs().max():.1e}")

39 Läufe, 546 Snapshots
logvar nicht exakt bei 0,01: 108 von 546 Snapshots; davon mit β > 0: 59, größtes β/d dort 6.2e-04, Epochen 120–240
Zerlegung Rate = Schranke + Rest: in jedem Snapshot exakt geprüft (assert)
T(0,01·μ) − T(μ) bei k = 20, Laufende, maximal: 0.0e+00


**Pro Dimension: SNR gegen Rate.** Jeder Punkt ist eine Dimension in einem Snapshot (β > 0). Kollabierte Dimensionen nach ε-δ
(KL < 0,01 Nats bei ≥ 99 % der Patienten) sind rot. Gestrichelt: die Informationsschranke ½ log(1 + SNR); der Abstand
zwischen Punkt und Linie ist Rate ohne Information. Senkrecht: SNR = 1 und die Burda-Schwelle A_u = 10⁻² (bei σ² ≈ 1,01
gleich SNR ≈ 0,0099).

In [10]:
per_dim = []
for row in snr[snr.beta_over_d > 0].itertuples():
    folder = RUNS / row.name
    valid = load_snapshot(folder, row.epoch, "valid")
    kl = compute_conditional_kl_matrix(valid["mu"], valid["logvar"])
    collapsed = np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta
    for dim in range(LATENT_DIM):
        per_dim.append(dict(name=row.name, epoch=row.epoch, dim=dim, snr=getattr(row, f"snr_{dim}"),
                            rate=getattr(row, f"rate_{dim}"), collapsed=bool(collapsed[dim])))
per_dim = pd.DataFrame(per_dim)
fig, ax = plt.subplots(figsize=(8, 5))
floor = 1e-6
for flag, color, label in ((False, "#1f77b4", "aktiv"), (True, "#c0392b", "kollabiert (ε-δ)")):
    part = per_dim[per_dim.collapsed.eq(flag)]
    ax.scatter(part.snr.clip(lower=floor), part.rate.clip(lower=floor), s=8, alpha=.5, color=color, label=label)
grid_snr = np.logspace(-6, 3.5, 200)
ax.plot(grid_snr, 0.5 * np.log1p(grid_snr), "k--", lw=1, label="Schranke ½ log(1 + SNR)")
ax.axvline(1, color="gray", ls=":", lw=.8)
ax.axvline(1e-2 / 1.01, color="orange", ls=":", lw=.8, label="Burda: A_u = 10⁻²")
ax.set(xscale="log", yscale="log", xlabel="SNR_d", ylabel="Rate_d [Nats]", title="Dimensionen: Signal/Rauschen gegen Rate")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "snr_vs_rate_per_dimension.png", dpi=150)
plt.show()
active = per_dim[~per_dim.collapsed]
display(pd.DataFrame({"kollabiert: max SNR": [per_dim[per_dim.collapsed].snr.max()],
                      "aktiv: min SNR": [active.snr.min()], "aktiv: 1 %-Quantil SNR": [active.snr.quantile(.01)],
                      "Dimensionen×Snapshots": [len(per_dim)]}).style.format("{:.3g}")
        .set_caption("SNR kollabierter und aktiver Dimensionen (gemessen, nicht hergeleitet)"))
BURDA_SNR = 1e-2 / np.exp(0.01)  # A_u = Var(μ) ≤ 10⁻² (Burda et al. 2016) bei σ² = e^0,01
dying = per_dim[~per_dim.collapsed & (per_dim.snr <= BURDA_SNR)]
print(f"{len(dying)} Dimension×Snapshot-Paare sind nach Burda inaktiv, nach ε-δ aber nicht kollabiert: eine Minderheit der "
      "Patienten hat noch μ > 0 (δ = 1 %), die Dimension trägt kaum Signal (sterbende Dimension).")

,kollabiert: max SNR,aktiv: min SNR,aktiv: 1 %-Quantil SNR,Dimensionen×Snapshots
0,0.00115,0.0012,0.0166,3.79e+03


26 Dimension×Snapshot-Paare sind nach Burda inaktiv, nach ε-δ aber nicht kollabiert: eine Minderheit der Patienten hat noch μ > 0 (δ = 1 %), die Dimension trägt kaum Signal (sterbende Dimension).


**Trustworthiness gegen die gemeinsame Informationsschranke, pro k.** Links T(μ), rechts E[T(z)]. Jeder Punkt ist ein
Snapshot (β > 0), die Farbe ist k. Die Tabelle darunter: lineare Anpassung T = a + b·x mit x = Schranke bzw. log(Schranke),
R² und Spearman ρ. Beschreibend; wegen der Skaleninvarianz ist für T(μ) kein fester Zusammenhang zu erwarten, für E[T(z)]
ein monotoner, aber nicht notwendig linearer.

In [11]:
from scipy.stats import spearmanr

fit_data = snr[snr.beta_over_d > 0]
k_colors = dict(zip(K_SWEEP, plt.cm.plasma(np.linspace(0, .85, len(K_SWEEP)))))
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for axis, space, title in ((axes[0], "mu", "T(μ)"), (axes[1], "z", "E[T(z)]")):
    for k in K_SWEEP:
        axis.scatter(fit_data.information_bound_joint, fit_data[f"t_{space}_k{k}"], s=10, alpha=.6, color=k_colors[k], label=f"k = {k}")
    axis.set(xscale="log", xlabel="I(n; z) ≤ ½ log det(I + Cov(μ)/σ²) [Nats]", title=title)
axes[0].set_ylabel("Trustworthiness")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "trustworthiness_vs_information_bound.png", dpi=150)
plt.show()

fits = []
for space in ("mu", "z"):
    for k in K_SWEEP:
        part = fit_data.dropna(subset=[f"t_{space}_k{k}"])
        y = part[f"t_{space}_k{k}"].to_numpy()
        for predictor, values in (("Schranke", part.information_bound_joint.to_numpy()),
                                  ("log Schranke", np.log(part.information_bound_joint.to_numpy()))):
            slope, intercept = np.polyfit(values, y, 1)
            residual = y - (intercept + slope * values)
            fits.append(dict(Raum="μ" if space == "mu" else "z", k=k, Prädiktor=predictor, Steigung=slope, Achsenabschnitt=intercept,
                             R2=1 - residual.var() / y.var(), RMSE=np.sqrt(np.mean(residual ** 2)),
                             Spearman=spearmanr(values, y).statistic, n=len(y)))
fits = pd.DataFrame(fits)
fits.to_csv(OUTPUT / "trustworthiness_vs_bound_fits.csv", index=False)
display(fits.pivot_table(index=["Raum", "k"], columns="Prädiktor", values=["R2", "Steigung", "Spearman"])
        .style.format("{:.3f}").set_caption("T gegen Informationsschranke: lineare Anpassung pro k (beschreibend)"))

### 6.3 Wie zuverlässig ist die Collapse-Bestimmung?

Die KL pro Patient und Dimension ist geschlossen berechenbar, es gibt keinen Schätzfehler. Unsicher sein können nur die
Entscheidung (Schwellen ε, δ und die Stichprobe von 1.001 Validierungspatienten), der Split und der Zeitpunkt. Geprüft über
alle Dimension×Snapshot-Paare mit β > 0:

1. **Abstand zur Entscheidungsgrenze:** ε-δ nennt eine Dimension kollabiert, wenn höchstens δ·n = 10 Patienten KL ≥ ε haben.
   Wie viele Patienten liegen bei kollabierten bzw. aktiven Dimensionen tatsächlich über ε?
2. **Schwelle ε:** Anzahl kollabierter Paare für ε = 10⁻⁴ … 10⁻¹ (wie Lucas et al. 2019, Abb. 5).
3. **Split:** Übereinstimmung Validierung ↔ Training.
4. **Kriterien:** ε-δ gegen Active Units nach Burda et al. (2016).
5. **Zeitpunkt:** Wechsel zwischen aufeinanderfolgenden Snapshots (20 Epochen).

In [12]:
reliability = []
for spec in done:
    folder = RUNS / spec.name
    log = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    for path in sorted((folder / "epoch_snapshots").glob("epoch_*_valid.npz")):
        epoch = int(path.stem.split("_")[1])
        if log.loc[epoch, "beta_over_d"] <= 0:
            continue
        kl = {split: compute_conditional_kl_matrix(load_snapshot(folder, epoch, split)["mu"],
                                                   load_snapshot(folder, epoch, split)["logvar"]) for split in ("valid", "train")}
        mu = load_snapshot(folder, epoch, "valid")["mu"]
        for dim in range(LATENT_DIM):
            row = dict(name=spec.name, epoch=epoch, dim=dim, above=int((kl["valid"][:, dim] >= COLLAPSE.primary_epsilon).sum()),
                       collapsed=np.mean(kl["valid"][:, dim] < COLLAPSE.primary_epsilon) >= 1 - COLLAPSE.delta,
                       collapsed_train=np.mean(kl["train"][:, dim] < COLLAPSE.primary_epsilon) >= 1 - COLLAPSE.delta,
                       burda_inactive=mu[:, dim].var() <= 1e-2)
            row |= {f"collapsed_eps_{eps:g}": np.mean(kl["valid"][:, dim] < eps) >= 1 - COLLAPSE.delta for eps in (1e-4, 1e-3, 1e-2, 1e-1)}
            reliability.append(row)
reliability = pd.DataFrame(reliability)
n_valid = len(VALID_IDS)
limit = int(COLLAPSE.delta * n_valid)
collapsed_rows, active_rows = reliability[reliability.collapsed], reliability[~reliability.collapsed]
print(f"{len(reliability)} Dimension×Snapshot-Paare (β > 0), davon nach ε-δ kollabiert: {len(collapsed_rows)}")
print(f"1. Grenze ≤ {limit} Patienten über ε. Kollabiert: höchstens {collapsed_rows.above.max()} Patienten über ε; "
      f"aktiv: mindestens {active_rows.above.min()}, 1 %-Quantil {active_rows.above.quantile(.01):.0f}; "
      f"Paare mit 1 … {3 * limit} Patienten über ε: {int(reliability.above.between(1, 3 * limit).sum())}")
print("2. Kollabierte Paare je ε:", {eps: int(reliability[f"collapsed_eps_{eps:g}"].sum()) for eps in (1e-4, 1e-3, 1e-2, 1e-1)})
print(f"3. Validierung ↔ Training gleich: {(reliability.collapsed == reliability.collapsed_train).mean():.2%} "
      f"({int((reliability.collapsed != reliability.collapsed_train).sum())} Abweichungen)")
display(pd.crosstab(reliability.collapsed.rename("ε-δ kollabiert"), reliability.burda_inactive.rename("Burda inaktiv (A_u ≤ 10⁻²)")))
ordered = reliability.sort_values(["name", "dim", "epoch"])
previous = ordered.groupby(["name", "dim"]).collapsed.shift()
known = previous.notna()
print(f"5. Wechsel zwischen Snapshots: stirbt {int((~previous[known].astype(bool) & ordered.collapsed[known]).sum())}, "
      f"lebt wieder auf {int((previous[known].astype(bool) & ~ordered.collapsed[known]).sum())}")

7440 Dimension×Snapshot-Paare (β > 0), davon nach ε-δ kollabiert: 654
1. Grenze ≤ 10 Patienten über ε. Kollabiert: höchstens 9 Patienten über ε; aktiv: mindestens 17, 1 %-Quantil 54; Paare mit 1 … 30 Patienten über ε: 27
2. Kollabierte Paare je ε: {0.0001: 635, 0.001: 639, 0.01: 654, 0.1: 677}
3. Validierung ↔ Training gleich: 99.89% (8 Abweichungen)


Burda inaktiv (A_u ≤ 10⁻²),False,True
ε-δ kollabiert,,
False,6741,45
True,0,654


5. Wechsel zwischen Snapshots: stirbt 47, lebt wieder auf 3


## 7. Frage 5: Vom Cluster zum Overlap

Labelfrei: k-means auf X (Controller-Subset) definiert die Referenz-Cluster. Gezeigt ist pro Sollwert (Seed 17) die UMAP
von z am Laufende (AUTOENCODIX-Stil, eingefärbt nach Krebsart) und darunter der automatisch erzeugte Overlap-Text.

In [13]:
show_targets = [t for t in (0.95, 0.90, 0.80, 0.70) if f"A_T{t:.3f}_k20_s17" in results.index]
fig, axes = plt.subplots(1, len(show_targets), figsize=(6 * len(show_targets), 4.6), squeeze=False)
for axis, target in zip(axes[0], show_targets):
    name = f"A_T{target:.3f}_k20_s17"
    dashboard = LiveDashboard(RUNS / name)
    dashboard.update_metrics()  # bindet den Lauf-Ordner, Kennzahlen kommen aus dem Cache
    axis.imshow(dashboard.umap_image(EPOCHS))
    axis.set_axis_off()
    row = results.loc[name]
    axis.set_title(f"Ziel {target:g}: Rate {row.rate:.2f} Nats, eigener Cluster {row.same_cluster_z:.0%}", fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT / "umap_by_target.png", dpi=150)
plt.show()
for target in show_targets:
    name = f"A_T{target:.3f}_k20_s17"
    texts = json.loads((RUNS / name / "live_descriptions.json").read_text())
    display(Markdown(f"**Ziel {target:g}** (Epoche {EPOCHS})\n\n" + texts[str(EPOCHS)].replace("\n", "\n\n")))

**Ziel 0.95** (Epoche 600)

X hat 15 Cluster (k-means, beste Silhouette 0,11: schwache Clusterstruktur); der Latentraum selbst trägt 19 Cluster (Silhouette 0,38: mittlere Clusterstruktur).

Von den 20 nächsten Nachbarn eines Patienten liegen im Latentraum 77 % im eigenen X-Cluster (in X: 70 %, Zufall: 9 %). Der Overlap ist also 23 % statt 30 %.

Die X-Cluster sind im Latentraum mit ARI 0,52 wiederzufinden (1 = identisch, 0 = Zufall); Silhouette der X-Cluster im Latentraum 0,27.

Am stärksten zusammengewachsen: 6 (UCEC 92%, BRCA 4%) ↔ 8 (OV 66%, UCEC 16%): 10 % gemeinsame Nachbarn (X: 5 %); 4 (LGG 88%, GBM 12%) ↔ 10 (PCPG 34%, SARC 29%): 9 % gemeinsame Nachbarn (X: 4 %); 2 (SKCM 72%, UVM 24%) ↔ 10 (PCPG 34%, SARC 29%): 3 % gemeinsame Nachbarn (X: 1 %).

Am stärksten aufgelöst: 4 (LGG 88%, GBM 12%) behält 86 % Nachbarn im eigenen Cluster (X: 91 %); 11 (HNSC 23%, LUAD 19%) behält 72 % Nachbarn im eigenen Cluster (X: 75 %).

**Ziel 0.9** (Epoche 600)

X hat 15 Cluster (k-means, beste Silhouette 0,11: schwache Clusterstruktur); der Latentraum selbst trägt 20 Cluster (Silhouette 0,52: deutliche Clusterstruktur).

Von den 20 nächsten Nachbarn eines Patienten liegen im Latentraum 71 % im eigenen X-Cluster (in X: 70 %, Zufall: 9 %). Der Overlap ist also 29 % statt 30 %.

Die X-Cluster sind im Latentraum mit ARI 0,42 wiederzufinden (1 = identisch, 0 = Zufall); Silhouette der X-Cluster im Latentraum 0,23.

Am stärksten zusammengewachsen: 0 (COAD 31%, STAD 29%) ↔ 7 (COAD 57%, READ 29%): 28 % gemeinsame Nachbarn (X: 14 %); 10 (PCPG 34%, SARC 29%) ↔ 12 (BLCA 22%, LUSC 21%): 18 % gemeinsame Nachbarn (X: 5 %); 10 (PCPG 34%, SARC 29%) ↔ 14 (KIRC 46%, KIRP 16%): 19 % gemeinsame Nachbarn (X: 8 %).

Am stärksten aufgelöst: 3 (BRCA 98%, LIHC 2%) behält 70 % Nachbarn im eigenen Cluster (X: 90 %); 1 (THCA 95%, LGG 5%) behält 79 % Nachbarn im eigenen Cluster (X: 94 %).

**Ziel 0.8** (Epoche 600)

X hat 15 Cluster (k-means, beste Silhouette 0,11: schwache Clusterstruktur); der Latentraum selbst trägt 5 Cluster (Silhouette 0,56: deutliche Clusterstruktur).

Von den 20 nächsten Nachbarn eines Patienten liegen im Latentraum 43 % im eigenen X-Cluster (in X: 70 %, Zufall: 9 %). Der Overlap ist also 57 % statt 30 %.

Die X-Cluster sind im Latentraum mit ARI 0,28 wiederzufinden (1 = identisch, 0 = Zufall); Silhouette der X-Cluster im Latentraum -0,02.

Am stärksten zusammengewachsen: 0 (COAD 31%, STAD 29%) ↔ 7 (COAD 57%, READ 29%): 36 % gemeinsame Nachbarn (X: 14 %); 3 (BRCA 98%, LIHC 2%) ↔ 11 (HNSC 23%, LUAD 19%): 25 % gemeinsame Nachbarn (X: 4 %); 1 (THCA 95%, LGG 5%) ↔ 14 (KIRC 46%, KIRP 16%): 15 % gemeinsame Nachbarn (X: 2 %).

Am stärksten aufgelöst: 3 (BRCA 98%, LIHC 2%) behält 15 % Nachbarn im eigenen Cluster (X: 90 %); 13 (PRAD 97%, PCPG 3%) behält 18 % Nachbarn im eigenen Cluster (X: 87 %).

**Ziel 0.7** (Epoche 600)

X hat 15 Cluster (k-means, beste Silhouette 0,11: schwache Clusterstruktur); der Latentraum selbst trägt 2 Cluster (Silhouette 0,79: deutliche Clusterstruktur).

Von den 20 nächsten Nachbarn eines Patienten liegen im Latentraum 27 % im eigenen X-Cluster (in X: 70 %, Zufall: 9 %). Der Overlap ist also 73 % statt 30 %.

Die X-Cluster sind im Latentraum mit ARI 0,15 wiederzufinden (1 = identisch, 0 = Zufall); Silhouette der X-Cluster im Latentraum -0,41.

Am stärksten zusammengewachsen: 8 (OV 66%, UCEC 16%) ↔ 12 (BLCA 22%, LUSC 21%): 25 % gemeinsame Nachbarn (X: 3 %); 8 (OV 66%, UCEC 16%) ↔ 11 (HNSC 23%, LUAD 19%): 26 % gemeinsame Nachbarn (X: 5 %); 7 (COAD 57%, READ 29%) ↔ 8 (OV 66%, UCEC 16%): 17 % gemeinsame Nachbarn (X: 0 %).

Am stärksten aufgelöst: 3 (BRCA 98%, LIHC 2%) behält 1 % Nachbarn im eigenen Cluster (X: 90 %); 7 (COAD 57%, READ 29%) behält 0 % Nachbarn im eigenen Cluster (X: 77 %).

## 8. Frage 6: k des Controllers

Gleiches Ziel 0,90, gesteuert mit Trustworthiness bei k = 10, 20, 40 oder 200. Verglichen werden End-β/d, Rate und die
Erhaltung bei festen k (unabhängig vom Steuer-k).

In [14]:
B = results[results.target.eq(0.90) & results.block.isin(["A", "B"])]
columns = ["beta_over_d", "rate", "rate_saving", "max_collapsed_dimensions", "t_mu_k20", "rnx_z_k10", "rnx_z_k40",
           "rnx_z_k200", "auc_cancer", "auc_sex_mixed", "same_cluster_z"]
k_summary = B.groupby("k")[columns].median()
display(k_summary.style.format("{:.3f}").format({"beta_over_d": "{:.2e}", "rate_saving": "{:.1%}"})
        .set_caption("Ziel 0,90: Median über Seeds je Controller-k"))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for k, group in B.groupby("k"):
    path = trajectories[trajectories.name.isin(group.index) & trajectories.seed.eq(17)]
    log = pd.read_csv(RUNS / group.index[group.seed.eq(17).to_numpy()][0] / "controller_log.csv") if (group.seed == 17).any() else None
    if log is not None:
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", label=f"k = {k}")
    axes[1].plot(path.epoch, path.rate, "o-", ms=3, label=f"k = {k}")
    axes[2].plot(path.epoch, path.trustworthiness, "o-", ms=3, label=f"k = {k}")
axes[0].set(yscale="log", title="β/d (Seed 17)", xlabel="Epoche")
axes[1].set(yscale="log", title="Rate", xlabel="Epoche")
axes[2].set(title="Trustworthiness bei k = 20", xlabel="Epoche")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "controller_k.png", dpi=150)
plt.show()

,beta_over_d,rate,rate_saving,max_collapsed_dimensions,t_mu_k20,rnx_z_k10,rnx_z_k40,rnx_z_k200,auc_cancer,auc_sex_mixed,same_cluster_z
k,,,,,,,,,,,
10,9.69e-04,3.381194,99.9%,0.000000,0.896894,0.055048,0.119251,0.193022,0.822934,0.571426,0.270937
20,1.08e-03,3.162333,99.9%,0.000000,0.900573,0.055934,0.116916,0.180942,0.810632,0.575129,0.272875
40,9.69e-04,3.751405,99.9%,0.000000,0.906302,0.065302,0.130140,0.201075,0.832633,0.572000,0.298500
200,0.00e+00,6842.933931,-226.8%,0.000000,0.942288,0.456827,0.491644,0.434231,0.989951,0.950893,0.663625


## 9. Einordnung

39 Läufe: TCGA, d = 8, 600 Epochen, Seeds 17, 23 und 31. Kennzeichnung: **gemessen** (diese Läufe), **hergeleitet**
(Rechnung oder Code), **Vermutung** (nicht geprüft). Zahlen sind Mediane über Seeds, sofern nicht anders angegeben.

**1. Collapse.** *Gemessen:* Bei den Zielen 0,95 bis 0,875 (k = 20) gibt es in keinem der drei Seeds eine nach ε-δ
kollabierte oder nach Burda inaktive Dimension. Bei 0,85 kollabiert ein Seed von drei, bei 0,80 zwei, ab 0,75 alle drei.
Trustworthiness auf μ steht dabei am Sollwert: Der Controller bemerkt den Collapse nicht. *Hergeleitet:* T hängt nur von
Rängen ab und ist skaleninvariant (T(0,01·μ) = T(μ), Abschnitt 6.2); die Größe, die über Collapse entscheidet, ist
SNR = Var(μ)/σ². *Gemessen:* Ohne Regelung kollabiert β/d = 1,5·10⁻³ ebenfalls nicht und landet mit Rate 2,3 Nats und
T = 0,885 dort, wo der Annealer mit Ziel 0,875 landet. Der Annealer verhindert Collapse also nicht besser als ein passend
gewähltes festes β; sein Nutzen ist, dieses β ohne Vorwissen zu finden. β/d = 4·10⁻³ ohne Regelung kollabiert in allen
drei Seeds (Median 4 Dimensionen).

**2. Rate.** *Gemessen:* Rate am Laufende 14,1 (Ziel 0,95), 5,6 (0,925), 3,2 (0,90), 2,5 Nats (0,875, letzte Stufe ohne
Collapse). Gegenüber dem Warm-up-Ende (β = 0, etwa 3.200 Nats) sind das über 99 % weniger; diese Zahl ist nur eine
Größenordnung, weil μ bei β = 0 ungebremst wächst.

**3. Preservation unter Druck.** T(μ) folgt dem Sollwert per Konstruktion und taugt hier nicht als unabhängige Antwort.
*Gemessen:* R_NX auf z fällt mit der Rate stetig, ohne Plateau (k = 10: 0,19 bei 14 Nats, 0,06 bei 3,2, 0,008 bei
0,66 Nats). E[T(z)] hängt bei allen k fast monoton an der Informationsschranke (Spearman ≥ 0,995, R² gegen log Schranke
0,96; beschreibend, die Snapshots sind nicht unabhängig). Auf μ bleibt der Anteil der Nachbarn im eigenen X-Cluster bis
Ziel 0,90 auf oder über dem Wert in X (0,74–0,80 gegen 0,70), auf z fällt er von 0,53 (0,95) auf 0,12 (0,70). Bei wenig
Druck bleibt also die Struktur von μ erhalten, die Nachbarschaften in z verlieren sich durch das Rauschen σ = 1 aber
schon früh.

**4. AUC auf z.** *Gemessen:* Krebs-AUC 0,984 am Warm-up-Ende, 0,963 (Ziel 0,95), 0,81 (0,90), 0,79 (0,875), 0,62 (0,70).
Die Geschlechts-AUC innerhalb gemischter Krebsarten fällt schon beim schwächsten Druck von 0,91 auf 0,64 (0,95) und
liegt darunter bei 0,53–0,58: Geschlecht geht zuerst verloren, Krebsart graduell.

**5. Raum.** *Gemessen (labelfrei, Seed 17):* Der Latentraum selbst trägt 19–20 Cluster bei Ziel 0,95 und 0,90, 5 bei
0,80 und 2 bei 0,70; die X-Cluster sind mit ARI 0,52 → 0,14 wiederzufinden (Median über Seeds). Der Übergang von
getrennten Clustern zu Overlap passiert zwischen 0,90 und 0,80, also dort, wo auch Dimensionen sterben.

**6. k des Controllers.** *Gemessen:* Bei Ziel 0,90 landen k = 10, 20 und 40 bei Rate 3,2–3,8 Nats ohne Collapse. Mit
k = 200 liegt T schon am Warm-up-Ende unter dem absoluten Ziel 0,90; der Controller erhöht β nie. Ein fairer Vergleich
über k braucht relative Ziele.

**Zuverlässigkeit der Collapse-Bestimmung** (Abschnitt 6.3, *gemessen*): Kollabierte Dimensionen haben höchstens 9 von
1.001 Patienten über ε, aktive mindestens 17; die Zahl kollabierter Paare ändert sich über ε = 10⁻⁴ … 10⁻¹ um weniger
als ±5 %; Validierung und Training stimmen in 99,9 % überein. Unsicher ist nicht die Messung, sondern das Training:
Seeds und Zeitpunkt (47 Übergänge aktiv → tot, 3 zurück).

**Varix-spezifisch.** *Hergeleitet und gemessen:* σ² ist durch die logvar-Klemme praktisch fest (Abschnitt 6.2), die
μ-Klemme setzt tote Dimensionen exakt auf 0. Dimensionen sterben auch ohne KL-Druck (β = 0, Lauf k = 200) und können
zurückkommen. Ein Teil des Collapse kann deshalb von der Klemme kommen und nicht vom Rate-Druck; das trennen diese
Läufe nicht.

**Offen.** *Vermutung:* Eine zweite Nebenbedingung auf SNR (min_d SNR_d ≥ s) oder Steuerung auf E[T(z)] statt T(μ)
würde Collapse direkt verhindern, weil beide über μ/σ am SNR hängen. Nicht geprüft; ebenso relative Ziele für den
k-Vergleich und andere d und Datensätze.

## 10. Zusatzversuch: Wird der Code besser, wenn der Annealer β hält?

**Frage.** Wenn der Annealer β nach Erreichen des Sollwerts lange hält: Findet das Modell eine Repräsentation, die mit
weniger Information auskommt (Rate sinkt) oder besser rekonstruiert, etwa weil ähnliche Cluster zusammenrücken? Und kann
der Controller den gewonnenen Spielraum nutzen und β weiter erhöhen?

**Was die Theorie zulässt.**
* *Quelle:* Bei festem β minimiert das Training D + β·R. Auf der optimalen Rate-Distortion-Kurve lassen sich D und R bei
  festem β nicht gleichzeitig verbessern; ein trainierter VAE liegt aber im Allgemeinen nicht auf ihr, Modelle mit gleichem
  ELBO können an verschiedenen (R, D)-Punkten liegen (Alemi et al. 2018, *Fixing a Broken ELBO*, arXiv:1711.00464). Dazu
  kommen Amortisations- und Approximationslücke des Encoders (Cremer, Li & Duvenaud 2018, *Inference Suboptimality in
  Variational Autoencoders*, ICML). Beide können beim Halten gleichzeitig sinken.
* *Quelle:* Die Idee einer Kompressionsphase bei unverändertem Ziel stammt aus Shwartz-Ziv & Tishby 2017 (arXiv:1703.00810);
  Saxe et al. 2018 (*On the Information Bottleneck Theory of Deep Learning*, ICLR) finden sie nur bei sättigenden
  Aktivierungen, nicht bei ReLU. Ob das auf die ReLU-artige μ-Klemme von Varix übertragbar ist, ist *Vermutung*.
* *Hergeleitet:* σ² liegt an der Klemme fest (Abschnitt 6.2). Die Rate kann daher nur sinken, wenn die μ zusammenrücken
  (Var(μ) oder m² kleiner). Das senkt die Informationsschranke ½ log(1 + SNR) und tendenziell auch die Trennung der
  Nachbarschaften; Trustworthiness ist aber rangbasiert und kann dabei gleich bleiben.

**Vorab festgelegte Kriterien.** *Effizienterer Code:* Rate oder Informationsschranke sinken während der Haltephase,
während Rekonstruktion nicht schlechter wird und T im Band bleibt. *Cluster rücken zusammen:* Anteil der z-Nachbarn im
eigenen X-Cluster und ARI sinken, ohne dass Dimensionen kollabieren. *Spielraum genutzt:* Im Arm mit aktivem Controller
steigt β nach dem ersten Erreichen des Bands weiter.

### 10.1 Rückblick: Haltephasen in den 600-Epochen-Läufen

Pro Lauf die letzte Strecke mit konstantem β (mindestens 100 Epochen, nur β > 0). Verglichen werden die ersten und die
letzten 50 Epochen dieser Strecke: Rekonstruktion (nativer Validation- und Train-Loss), Rate (alle Validierungspatienten,
Controller-Log) und Trustworthiness.

In [7]:
def constant_beta_tail(log):
    """Erste Epoche nach dem Warm-up, ab der β bis zum Laufende unverändert bleibt."""
    beta = log.set_index("epoch").effective_beta
    beta = beta[beta.index > CONTROLLER["warmup_epochs"]]
    changes = beta.index[1:][~np.isclose(beta.values[1:], beta.values[:-1], rtol=1e-9, atol=0)]
    return int(changes.max()) if len(changes) else int(beta.index.min())


def window_change(series, first, last):
    return float(series.loc[last].mean() / series.loc[first].mean() - 1)


rows = []
for spec in done:
    folder = RUNS / spec.name
    log = pd.read_csv(folder / "controller_log.csv")
    losses = pd.read_csv(folder / "epoch_losses.csv")
    start, end = constant_beta_tail(log), int(log.epoch.max())
    beta_over_d = log.set_index("epoch").beta_over_d.loc[end]
    if end - start + 1 < 100 or beta_over_d == 0:
        continue
    first, last = slice(start, start + 49), slice(end - 49, end)
    controller = log.set_index("epoch")
    recon = {split: losses[losses.split.eq(split)].set_index("epoch").recon_loss for split in ("valid", "train")}
    rows.append(dict(name=spec.name, block=spec.block, target=spec.target, seed=spec.seed, hold_from=start,
                     hold_epochs=end - start + 1, beta_over_d=beta_over_d,
                     recon_valid=window_change(recon["valid"], first, last),
                     recon_train=window_change(recon["train"], first, last),
                     rate=window_change(controller.rate, first, last),
                     rate_sd_first_window=float(controller.rate.loc[first].std() / controller.rate.loc[first].mean()),
                     t_change=float(controller.metric_value.loc[last].mean() - controller.metric_value.loc[first].mean()),
                     collapsed=f"{controller.collapsed_dimensions.loc[start]} → {controller.collapsed_dimensions.loc[end]}"))
retro = pd.DataFrame(rows).sort_values(["block", "target", "seed"], ascending=[True, False, True])
display(retro.style.format({"beta_over_d": "{:.1e}", "recon_valid": "{:+.2%}", "recon_train": "{:+.2%}", "rate": "{:+.2%}",
                            "rate_sd_first_window": "{:.2%}", "t_change": "{:+.3f}"})
        .set_caption("Haltephasen ≥ 100 Epochen bei β > 0: letzte 50 gegen erste 50 Epochen"))
print(f"{len(retro)} Läufe: Rekonstruktion (Validation) sinkt in {int((retro.recon_valid < 0).sum())}, "
      f"Rate steigt in {int((retro.rate > 0).sum())}, beide sinken in "
      f"{int(((retro.recon_valid < 0) & (retro.rate < 0)).sum())}; |ΔT| ≤ {retro.t_change.abs().max():.3f}")
retro.to_csv(OUTPUT / "hold_retrospective.csv", index=False)

,name,block,target,seed,hold_from,hold_epochs,beta_over_d,recon_valid,recon_train,rate,rate_sd_first_window,t_change,collapsed
0,A_T0.950_k20_s17,A,0.950000,17,226,375,2.0e-04,-2.86%,-4.66%,+2.41%,3.11%,-0.004,0 → 0
7,A_T0.950_k20_s23,A,0.950000,23,221,380,2.0e-04,-3.55%,-4.82%,+3.30%,3.74%,-0.009,0 → 0
14,A_T0.950_k20_s31,A,0.950000,31,366,235,1.0e-04,-1.55%,-2.52%,+0.57%,2.86%,-0.001,0 → 0
1,A_T0.925_k20_s17,A,0.925000,17,386,215,8.7e-04,-0.62%,-0.78%,+1.50%,3.53%,+0.001,0 → 0
8,A_T0.925_k20_s23,A,0.925000,23,391,210,6.9e-04,-0.76%,-0.85%,+2.01%,3.09%,+0.003,0 → 0
15,A_T0.925_k20_s31,A,0.925000,31,236,365,6.2e-04,-2.18%,-2.24%,+1.59%,3.93%,-0.005,0 → 0
2,A_T0.900_k20_s17,A,0.900000,17,321,280,1.2e-03,-0.11%,-1.04%,+0.66%,3.61%,-0.001,0 → 0
9,A_T0.900_k20_s23,A,0.900000,23,396,205,1.1e-03,-0.54%,-0.73%,+2.18%,3.44%,+0.005,0 → 0
16,A_T0.900_k20_s31,A,0.900000,31,311,290,1.1e-03,-1.05%,-1.26%,+1.36%,4.19%,-0.000,0 → 0
3,A_T0.875_k20_s17,A,0.875000,17,386,215,1.2e-03,-0.07%,-0.51%,-0.58%,3.88%,-0.001,0 → 0


20 Läufe: Rekonstruktion (Validation) sinkt in 18, Rate steigt in 16, beide sinken in 3; |ΔT| ≤ 0.009


`rate_sd_first_window` ist die Streuung der Rate (alle 5 Epochen) im ersten Fenster relativ zum Mittel: eine grobe
Rauschgröße, kein Test (aufeinanderfolgende Messungen sind korreliert). Die Haltephasen sind höchstens 380 Epochen lang
und beginnen je Lauf zu verschiedenen Zeitpunkten; für eine saubere Antwort folgt der Versuch unten.

### 10.2 Versuchsplan: 2.000 Epochen, Controller aktiv gegen eingefroren

Sollwert T = 0,90 (k = 20), sonst alles wie Block A, aber **2.000 Epochen**. Zwei Arme je Seed (17, 23, 31):

| Arm | Controller nach dem ersten Erreichen des Sollbands | beantwortet |
|---|---|---|
| H | regelt weiter (`MaximinAnnealer`, unverändert) | Nutzt der Controller gewonnenen Spielraum und erhöht β weiter? |
| F | friert β bei der ersten Messung im Band ein (`FrozenMaximinAnnealer`) | Was passiert bei exakt konstantem β mit Rekonstruktion, Rate, Struktur? |

Das Training ist bei gleichem Seed deterministisch: H und F sind bis zur ersten Messung im Band identisch, und bis
Epoche 595 auch identisch mit dem 600-Epochen-Lauf aus Block A (wird unten geprüft). Etwa 56 Minuten pro Lauf,
6 Läufe ≈ 6 Stunden; zuerst beide Arme mit Seed 17.

In [8]:
HOLD_EPOCHS, HOLD_TARGET, HOLD_K = 2000, 0.90, 20
RUN_HOLD_TRAINING = True


class FrozenMaximinAnnealer(MaximinAnnealer):
    """MaximinAnnealer, der β bei der ersten Messung im Sollband einfriert (Haltetest)."""

    def update(self, margins):
        if self.state == "frozen":
            return min(margins.values())
        signal = super().update(margins)
        if self.reason == "target_band":
            self.state, self.reason = "frozen", "first_target_band"
        return signal


@dataclass(frozen=True)
class HoldSpec:
    block: str  # "H": Controller regelt weiter, "F": β eingefroren
    seed: int
    target: float = HOLD_TARGET
    k: int = HOLD_K
    epochs: int = HOLD_EPOCHS

    @property
    def name(self):
        return f"{self.block}_T{self.target:.3f}_k{self.k}_e{self.epochs}_s{self.seed}"


# Test: nach der ersten Messung im Band ändert der eingefrorene Annealer β nicht mehr, der normale schon.
frozen, active = (cls([Constraint("t", None, .9, .01)]) for cls in (FrozenMaximinAnnealer, MaximinAnnealer))
for margin in (5, 5, 0.5, 5, 5, -5):
    frozen.update({"t": margin})
    active.update({"t": margin})
assert frozen.state == "frozen" and np.isclose(frozen.w, 1e-4 * 1.25) and not np.isclose(active.w, frozen.w)
print(f"Einfrieren geprüft: w eingefroren {frozen.w:.3g}, w aktiv {active.w:.3g}")

hold_grid = [HoldSpec(block, seed) for seed in SEEDS for block in ("H", "F")]
hold_handle = display(Markdown("_Noch keine Messung._"), display_id=True)


def hold_live_plot(spec):
    def draw(log):
        measured = log.dropna(subset=["metric_value"])
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", color="black")
        axes[0].set(yscale="log", title="β/d", xlabel="Epoche")
        axes[1].plot(measured.epoch, measured.metric_value, "o-", ms=2)
        axes[1].axhspan(spec.target - .01, spec.target + .01, alpha=.15)
        axes[1].set(title=f"Trustworthiness (k = {spec.k})", xlabel="Epoche")
        axes[2].plot(log.epoch, log.rate, lw=1.2)
        axes[2].set(yscale="log", title="Rate (alle Validierungspatienten)", xlabel="Epoche")
        fig.suptitle(spec.name)
        fig.tight_layout()
        hold_handle.update(fig)
        plt.close(fig)
    return draw


def train_hold(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    kind = FrozenMaximinAnnealer if spec.block == "F" else MaximinAnnealer
    annealer = kind([trustworthiness_constraint(spec.target, spec.k)], **CONTROLLER)
    config = build_varix_config(data_config, "tcga", dict(TRAINING, epochs=spec.epochs),
                                dict(RUN_DEFAULTS, beta=BETA_OVER_D_MAX * LATENT_DIM), global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, annealer=kind.__name__, beta_over_d_max=BETA_OVER_D_MAX, controller=CONTROLLER,
        n_subset=N_SUBSET, subset_seed=SUBSET_SEED, epochs=spec.epochs), indent=2, default=float) + "\n")
    loss_type = annealed_varix_loss(annealer, folder, VALID_IDS, POSITIONS, on_measure=hold_live_plot(spec))
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=SNAPSHOT_EVERY, downstream=False)
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    LiveDashboard(folder).update_metrics()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, train_seconds=trained,
                                                     total_seconds=time.time() - started), default=float) + "\n")


if RUN_HOLD_TRAINING:
    for spec in hold_grid:
        if not (RUNS / spec.name / "run.json").exists():
            print(f"{time.strftime('%H:%M')} Lauf {spec.name}", flush=True)
            train_hold(spec)
hold_done = [spec for spec in hold_grid if (RUNS / spec.name / "run.json").exists()]
print("fertig:", len(hold_done), "von", len(hold_grid))

Einfrieren geprüft: w eingefroren 0.000125, w aktiv 0.000175


<Figure size 1650x374 with 3 Axes>

10:58 Lauf H_T0.900_k20_e2000_s17


You are using a CUDA device ('NVIDIA GeForce RTX 4070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Reproducibility settings for device cuda are not implemented or necessary i.e. for cpu.
Epoch 1 - Train Loss: 0.0022
Sub-losses: recon_loss: 0.0022, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 1 - Valid Loss: 0.0010
Sub-losses: recon_loss: 0.0010, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 2 - Train Loss: 0.0011
Sub-losses: recon_loss: 0.0011, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 2 - Valid Loss: 0.0009
Sub-losses: recon_loss: 0.0009, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 3 - Train Loss: 0.0009
Sub-losses: recon_loss: 0.0009, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 3 - Valid Loss: 0.0009
Sub-losses: recon_loss: 0.0009, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 4 - Train Loss: 0.0008
Sub-losses: recon_loss: 0.0008, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_fac

### 10.3 Auswertung der Haltephase

**Haltephase** = ab der ersten Messung im Sollband (für F: ab dem Einfrieren). Laufend aus dem Controller-Log (alle
5 Epochen): β/d, Trustworthiness, Rate; aus dem nativen Loss-Log (jede Epoche): Rekonstruktion. Aus den Snapshots (Beginn
der Haltephase, dann alle 100 Epochen): Informationsschranke Σ_d ½ log(1 + SNR_d) und der Rest der Rate ohne Information
(Zerlegung aus Abschnitt 6.2), kollabierte Dimensionen, Anteil μ = 0, E[T(z)] (zwei z-Züge), AUC auf z (Krebsart,
Geschlecht in gemischten Krebsarten) und der labelfreie Cluster-Overlap (Anteil der z-Nachbarn im eigenen X-Cluster, ARI).
Die k-means-Clusterzahl wird bewusst nicht verwendet (Obergrenze 20, kann nie 1 ausgeben).

In [9]:
HOLD_SNAPSHOT_STEP = 100


def hold_start(log):
    """Erste trainierte Epoche nach der ersten Messung im Sollband."""
    reached = log[(log.epoch > CONTROLLER["warmup_epochs"]) & log.reason.isin(["target_band", "first_target_band"])]
    return int(reached.epoch.min()) if len(reached) else None


def information_split(mu, logvar):
    variance = np.exp(logvar)
    total = variance.mean(0) + mu.var(0)
    bound = 0.5 * (np.log(total) - np.log(variance).mean(0))
    return float(bound.sum()), float((0.5 * (mu.mean(0) ** 2 + total - 1 - np.log(total))).sum())


def analyse_hold(spec):
    folder = RUNS / spec.name
    cache = folder / "analysis_hold.csv"
    if cache.exists():
        return pd.read_csv(cache)
    log = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    live = pd.read_csv(folder / "live_metrics.csv").set_index("epoch")
    recon = pd.read_csv(folder / "epoch_losses.csv").query("split == 'valid'").set_index("epoch").recon_loss
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    start = hold_start(log.reset_index())
    first_snapshot = int(np.ceil(start / SNAPSHOT_EVERY) * SNAPSHOT_EVERY)
    epochs = sorted({CONTROLLER["warmup_epochs"], first_snapshot, spec.epochs}
                    | set(range(first_snapshot, spec.epochs + 1, HOLD_SNAPSHOT_STEP)))
    rows = []
    for epoch in epochs:
        valid = load_snapshot(folder, epoch, "valid")
        mu, logvar = valid["mu"].astype(np.float64), valid["logvar"].astype(np.float64)
        kl = compute_conditional_kl_matrix(mu, logvar)
        bound, no_information = information_split(mu, logvar)
        positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
        subset, sigma = mu[positions], np.exp(0.5 * logvar[positions])
        rng = np.random.default_rng(SUBSET_SEED)
        rows.append(dict(epoch=epoch, hold_start=start, beta_over_d=log.loc[epoch, "beta_over_d"],
                         recon_valid=float(recon.loc[epoch - 9:epoch].mean()), rate=float(kl.sum(1).mean()),
                         information_bound=bound, no_information=no_information,
                         collapsed_dimensions=int(np.sum(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)),
                         mu_zero_share=float((mu == 0).mean()), zero_patients=float((mu == 0).all(1).mean()),
                         t_mu=float(trustworthiness(x, subset, n_neighbors=20)) if len(np.unique(subset, axis=0)) > 20 else np.nan,
                         t_z=float(np.mean([trustworthiness(x, subset + sigma * rng.standard_normal(subset.shape), n_neighbors=20)
                                            for _ in range(Z_DRAWS)])),
                         **auc_on_z(folder, epoch), same_cluster_z=live.loc[epoch, "same_cluster_neighbours_z"],
                         same_cluster_mu=live.loc[epoch, "same_cluster_neighbours_latent"],
                         recovery_ari=live.loc[epoch, "recovery_ari"]))
    table = pd.DataFrame(rows)
    table.to_csv(cache, index=False)
    return table


hold = pd.concat([analyse_hold(spec).assign(name=spec.name, arm=spec.block, seed=spec.seed) for spec in hold_done],
                 ignore_index=True)

# Determinismus: bis Epoche 595 dasselbe β wie der 600-Epochen-Lauf aus Block A (gleicher Seed und Sollwert)
for spec in hold_done:
    reference = RUNS / RunSpec("A", spec.seed, spec.target).name / "controller_log.csv"
    if reference.exists():
        a = pd.read_csv(reference).set_index("epoch").effective_beta.loc[:595]
        h = pd.read_csv(RUNS / spec.name / "controller_log.csv").set_index("epoch").effective_beta.loc[:595]
        print(f"{spec.name}: β bis Epoche 595 identisch mit Block A: {np.allclose(a, h, rtol=1e-12, atol=0)}")

H_T0.900_k20_e2000_s17: β bis Epoche 595 identisch mit Block A: True
F_T0.900_k20_e2000_s17: β bis Epoche 595 identisch mit Block A: False
H_T0.900_k20_e2000_s23: β bis Epoche 595 identisch mit Block A: True
F_T0.900_k20_e2000_s23: β bis Epoche 595 identisch mit Block A: False
H_T0.900_k20_e2000_s31: β bis Epoche 595 identisch mit Block A: True
F_T0.900_k20_e2000_s31: β bis Epoche 595 identisch mit Block A: False


In [10]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
seed_colors = dict(zip(SEEDS, ("tab:blue", "tab:orange", "tab:green")))
for spec in hold_done:
    log = pd.read_csv(RUNS / spec.name / "controller_log.csv")
    losses = pd.read_csv(RUNS / spec.name / "epoch_losses.csv").query("split == 'valid'")
    table = hold[hold.name.eq(spec.name)]
    style = dict(color=seed_colors[spec.seed], ls="-" if spec.block == "H" else "--", lw=1.1,
                 label=f"{'aktiv' if spec.block == 'H' else 'eingefroren'}, Seed {spec.seed}")
    measured = log.dropna(subset=["metric_value"])
    axes[0, 0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", **style)
    axes[0, 1].plot(measured.epoch, measured.metric_value, **style)
    axes[0, 2].plot(log.epoch, log.rate, **style)
    axes[0, 3].plot(losses.epoch, losses.recon_loss.rolling(20).mean(), **style)
    for axis, column in zip(axes[1], ("information_bound", "no_information", "auc_cancer", "same_cluster_z")):
        axis.plot(table.epoch, table[column], "o", ms=3, **style)
    for axis in axes.ravel():
        axis.axvline(table.hold_start.iloc[0], color=seed_colors[spec.seed], lw=.5, alpha=.5)
titles = ["β/d", "Trustworthiness (Controller)", "Rate [Nats]", "Rekonstruktion Validation (gleitend 20)",
          "Informationsschranke Σ ½ log(1+SNR)", "Rate ohne Information", "Krebs-AUC (z)", "z-Nachbarn im eigenen X-Cluster"]
for axis, title in zip(axes.ravel(), titles):
    axis.set(title=title, xlabel="Epoche", xlim=(CONTROLLER["warmup_epochs"], HOLD_EPOCHS))
axes[0, 0].set_yscale("log")
axes[0, 2].set(yscale="log", ylim=(1, 30))
axes[0, 1].axhspan(HOLD_TARGET - .01, HOLD_TARGET + .01, alpha=.15)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Haltephase: durchgezogen Controller aktiv (H), gestrichelt eingefroren (F); senkrecht Beginn der Haltephase")
fig.tight_layout()
fig.savefig(OUTPUT / "hold_long.png", dpi=150)
plt.show()

In [11]:
def hold_summary(table, spec):
    start = table[table.epoch >= table.hold_start.iloc[0]].iloc[0]
    end = table.iloc[-1]
    log = pd.read_csv(RUNS / spec.name / "controller_log.csv").set_index("epoch")
    in_hold = log.loc[int(start.hold_start):]
    return dict(name=spec.name, arm="aktiv" if spec.block == "H" else "eingefroren", seed=spec.seed,
                hold_from=int(start.hold_start), beta_factor=end.beta_over_d / start.beta_over_d,
                beta_changes=int((in_hold.effective_beta.diff().abs() > 0).sum()),
                t_in_band=float((in_hold.metric_value.dropna().sub(HOLD_TARGET).abs() <= .01).mean()),
                recon=end.recon_valid / start.recon_valid - 1, rate=end.rate / start.rate - 1,
                information_bound=end.information_bound - start.information_bound,
                no_information=end.no_information - start.no_information,
                t_z=end.t_z - start.t_z, auc_cancer=end.auc_cancer - start.auc_cancer,
                auc_sex_mixed=end.auc_sex_mixed - start.auc_sex_mixed,
                same_cluster_z=end.same_cluster_z - start.same_cluster_z, recovery_ari=end.recovery_ari - start.recovery_ari,
                collapsed=f"{int(start.collapsed_dimensions)} → {int(end.collapsed_dimensions)}",
                mu_zero_share=end.mu_zero_share - start.mu_zero_share)


hold_table = pd.DataFrame([hold_summary(hold[hold.name.eq(spec.name)], spec) for spec in hold_done])
display(hold_table.style.format({"beta_factor": "{:.2f}×", "t_in_band": "{:.0%}", "recon": "{:+.2%}", "rate": "{:+.1%}",
                                 "information_bound": "{:+.3f}", "no_information": "{:+.3f}", "t_z": "{:+.3f}",
                                 "auc_cancer": "{:+.3f}", "auc_sex_mixed": "{:+.3f}", "same_cluster_z": "{:+.3f}",
                                 "recovery_ari": "{:+.3f}", "mu_zero_share": "{:+.1%}"})
        .set_caption("Änderung vom ersten Snapshot der Haltephase bis Epoche 2.000 (Rekonstruktion: Mittel der 10 Epochen davor)"))
hold_table.to_csv(OUTPUT / "hold_summary.csv", index=False)

,name,arm,seed,hold_from,beta_factor,beta_changes,t_in_band,recon,rate,information_bound,no_information,t_z,auc_cancer,auc_sex_mixed,same_cluster_z,recovery_ari,collapsed,mu_zero_share
0,H_T0.900_k20_e2000_s17,aktiv,17,261,0.72×,3,99%,-7.25%,+48.2%,+0.597,+0.548,+0.046,+0.039,+0.001,+0.064,+0.056,0 → 0,+1.1%
1,F_T0.900_k20_e2000_s17,eingefroren,17,261,1.00×,0,14%,-2.22%,+10.8%,+0.171,+0.087,+0.014,+0.004,-0.015,+0.016,-0.028,0 → 0,+2.7%
2,H_T0.900_k20_e2000_s23,aktiv,23,256,0.89×,1,100%,-3.27%,+11.8%,+0.253,+0.116,+0.022,+0.013,-0.021,+0.029,-0.032,0 → 0,+4.6%
3,F_T0.900_k20_e2000_s23,eingefroren,23,256,1.00×,0,99%,-1.51%,+2.6%,+0.124,-0.042,+0.013,+0.003,-0.020,+0.011,-0.076,0 → 0,+4.9%
4,H_T0.900_k20_e2000_s31,aktiv,31,261,0.85×,4,99%,-5.61%,+21.9%,+0.323,+0.222,+0.026,+0.021,-0.013,+0.024,+0.048,0 → 0,+3.0%
5,F_T0.900_k20_e2000_s31,eingefroren,31,261,1.00×,0,1%,-1.25%,-0.4%,+0.035,-0.044,+0.003,-0.004,-0.019,-0.014,-0.042,0 → 0,+2.9%


## 11. Zusatzversuch: erst mit β = 0 konvergieren, dann β stufenweise erhöhen

**Frage.** In Block A startet der Controller bei Epoche 100, also bevor das Modell konvergiert ist. Konvergiert mit β = 0
trägt z aber deutlich mehr Cluster-Struktur (`trustworthiness_09`, Abschnitt 3.4b: 13–30 HDBSCAN-Cluster, fast alle
rein eine Krebsart; am Warm-up-Ende nur 2). Was zerstört β, wenn es erst auf ein konvergiertes Modell trifft?

**Regel (Plateau-Leiter).** β/d läuft über eine feste Leiter, dieselben Werte wie der Sweep auf Folie 42 (dort je β ein
eigenes Modell) plus Zwischenstufen: 0 → 1,25·10⁻⁵ → 5·10⁻⁵ → 10⁻⁴ → 2,5·10⁻⁴ → 5·10⁻⁴ → 10⁻³ → 2·10⁻³ → 4·10⁻³.
Die nächste Stufe kommt erst, wenn sich die Validierungs-Rekonstruktion kaum noch ändert: Median der letzten 50 Epochen
gegenüber dem Median der 50 davor, relative Änderung ≤ 1 %. Mindestens 300 Epochen auf β = 0 und 100 auf jeder weiteren
Stufe; spätestens nach 1.000 (β = 0) bzw. 400 Epochen geht es trotzdem weiter (im Log als `max_epochs`). Geprüft wird
alle 20 Epochen, also genau an den Snapshot-Epochen: der letzte Snapshot einer Stufe ist ihr Plateau-Zustand.
Trustworthiness (k = 20) wird nur mitgeschrieben, sie steuert nichts. 3.000 Epochen, sonst alles wie Block A.

So entsteht in einem einzigen Lauf dieselbe β-Folge wie auf Folie 42, aber jedes β startet vom konvergierten Modell der
vorigen Stufe. Der Vergleich beider zeigt, ob der Weg eine Rolle spielt.

In [5]:
LADDER_LEVELS = (0.0, 1.25e-5, 5e-5, 1e-4, 2.5e-4, 5e-4, 1e-3, 2e-3, 4e-3)
LADDER = dict(window=50, tolerance=0.01, min_epochs_first=300, min_epochs=100, max_epochs_first=1000, max_epochs=400,
              every=SNAPSHOT_EVERY)
LADDER_EPOCHS = 3000
RUN_LADDER_TRAINING = True


class PlateauLadderAnnealer(MaximinAnnealer):
    """β/d über eine feste Leiter; nächste Stufe, sobald die Validierungs-Rekonstruktion ein Plateau erreicht."""

    def __init__(self, constraints, levels=LADDER_LEVELS, *, window, tolerance, min_epochs_first, min_epochs,
                 max_epochs_first, max_epochs, every):
        super().__init__(constraints, warmup_epochs=every, every=every)
        if levels[0] != 0 or any(b <= a for a, b in zip(levels, levels[1:])):
            raise ValueError("Leiter muss bei 0 beginnen und streng steigen.")
        if min(min_epochs_first, min_epochs) < 2 * window:
            raise ValueError("Mindestdauer einer Stufe muss zwei Fenster umfassen.")
        self.levels, self.window, self.tolerance = tuple(levels), window, tolerance
        self.min_epochs, self.max_epochs = (min_epochs_first, min_epochs), (max_epochs_first, max_epochs)
        self.level, self.level_start, self.recon, self.change = 0, 1, {}, np.nan
        self.state, self.reason = "level_0", ""

    def record(self, epoch, recon):
        """Validierungs-Rekonstruktion nach Epoche `epoch` (1-basiert)."""
        self.recon[epoch] = recon

    def update(self, margins):
        signal = min(margins.values())
        if self.level == len(self.levels) - 1:
            self.reason = "last_level"
            return signal
        epoch = max(self.recon)
        on_level = epoch - self.level_start + 1
        first = self.level == 0
        if on_level < self.min_epochs[not first]:
            self.reason = "min_epochs"
            return signal
        recent = pd.Series(self.recon).loc[epoch - 2 * self.window + 1:epoch].to_numpy()
        self.change = float(np.median(recent[self.window:]) / np.median(recent[:self.window]) - 1)
        plateau = abs(self.change) <= self.tolerance
        if plateau or on_level >= self.max_epochs[not first]:
            self.level += 1
            self.w = self.levels[self.level] / self.levels[-1]
            self.level_start = epoch + 1
            self.reason = "plateau" if plateau else "max_epochs"
        else:
            self.reason = "not_converged"
        self.state = f"level_{self.level}"
        return signal


def ladder_varix_loss(annealer, folder, valid_ids, positions, on_measure=None):
    """Wie `annealed_varix_loss`, gibt dem Annealer zusätzlich nach jeder Epoche die Validierungs-Rekonstruktion."""
    base = annealed_varix_loss(annealer, folder, valid_ids, positions, on_measure)

    class LadderVarixLoss(base):
        def __init__(self, config, annealing_scheduler=None):
            super().__init__(config, annealing_scheduler)
            self.recon_sum, self.recon_n = 0.0, 0

        def forward(self, model_output, targets, epoch=None, **kwargs):
            # Vor dem flush der Basisklasse (erste Trainings-Batch der nächsten Epoche) die Rekonstruktion abgeben.
            if epoch is not None and torch.is_grad_enabled() and epoch != self.epoch and self.recon_n:
                annealer.record(self.epoch + 1, self.recon_sum / self.recon_n)
                self.recon_sum, self.recon_n = 0.0, 0
            result = super().forward(model_output=model_output, targets=targets, epoch=epoch, **kwargs)
            if epoch is not None and not torch.is_grad_enabled():
                self.recon_sum += float(result[1]["recon_loss"]) * len(targets)
                self.recon_n += len(targets)
            return result

    return LadderVarixLoss


# Test mit künstlicher Rekonstruktion: fällt bis Epoche 400, dann flach; nach jeder Stufe kurzer Anstieg, der abklingt.
test = PlateauLadderAnnealer([Constraint("t", None, .9, .01)], **LADDER)
steps = []
for epoch in range(1, 2001):
    since = epoch - test.level_start
    test.record(epoch, 1e-3 * (1 + np.exp(-epoch / 80)) * (1 + 0.05 * test.level * np.exp(-since / 30)))
    if epoch % LADDER["every"] == 0:
        level = test.level
        test.update({"t": 0.0})
        if test.level != level:
            steps.append((epoch, test.level, test.reason))
assert steps[0][0] >= LADDER["min_epochs_first"] and all(b[0] - a[0] >= LADDER["min_epochs"] for a, b in zip(steps, steps[1:]))
assert [s[1] for s in steps] == list(range(1, len(LADDER_LEVELS))) and test.reason == "last_level"
print("Stufenwechsel (Epoche, Stufe, Grund):", steps)
stuck = PlateauLadderAnnealer([Constraint("t", None, .9, .01)], **LADDER)
for epoch in range(1, 1101):
    stuck.record(epoch, 1e-3 * 0.98 ** (epoch / 50))  # fällt stetig um 2 % je 50 Epochen, nie flach
    if epoch % LADDER["every"] == 0:
        stuck.update({"t": 0.0})
assert stuck.level == 1 and stuck.reason in ("max_epochs", "min_epochs", "not_converged") and stuck.level_start == 1001
print("Ohne Plateau: Stufe 1 ab Epoche", stuck.level_start, "(max_epochs auf β = 0)")

Stufenwechsel (Epoche, Stufe, Grund): [(380, 1, 'plateau'), (520, 2, 'plateau'), (660, 3, 'plateau'), (820, 4, 'plateau'), (980, 5, 'plateau'), (1160, 6, 'plateau'), (1340, 7, 'plateau'), (1520, 8, 'plateau')]
Ohne Plateau: Stufe 1 ab Epoche 1001 (max_epochs auf β = 0)


In [6]:
@dataclass(frozen=True)
class LadderSpec:
    seed: int
    epochs: int = LADDER_EPOCHS

    @property
    def name(self):
        return f"L_ladder_e{self.epochs}_s{self.seed}"


ladder_grid = [LadderSpec(seed) for seed in SEEDS]
ladder_handle = display(Markdown("_Noch keine Messung._"), display_id=True)


def ladder_live_plot(spec, annealer):
    def draw(log):
        measured = log.dropna(subset=["metric_value"])
        recon = pd.Series(annealer.recon)
        fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-7), where="post", color="black")
        axes[0].set(yscale="log", title=f"β/d (Stufe {annealer.level}, {annealer.reason})", xlabel="Epoche")
        axes[1].plot(recon.index, recon.values, lw=.6, color="grey")
        axes[1].plot(recon.index, recon.rolling(50).median(), color="black")
        axes[1].set(yscale="log", title=f"Rekonstruktion Validation (Änderung {annealer.change:+.2%})", xlabel="Epoche")
        axes[2].plot(measured.epoch, measured.metric_value, "o-", ms=2)
        axes[2].set(title="Trustworthiness (k = 20, nur mitgeschrieben)", xlabel="Epoche")
        axes[3].plot(log.epoch, log.rate, lw=1.2)
        axes[3].set(yscale="log", title="Rate (alle Validierungspatienten)", xlabel="Epoche")
        fig.suptitle(spec.name)
        fig.tight_layout()
        ladder_handle.update(fig)
        plt.close(fig)
    return draw


def train_ladder(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    annealer = PlateauLadderAnnealer([trustworthiness_constraint(0.9, 20)], **LADDER)
    config = build_varix_config(data_config, "tcga", dict(TRAINING, epochs=spec.epochs),
                                dict(RUN_DEFAULTS, beta=LADDER_LEVELS[-1] * LATENT_DIM), global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, annealer="PlateauLadderAnnealer", levels=LADDER_LEVELS, ladder=LADDER,
        n_subset=N_SUBSET, subset_seed=SUBSET_SEED, epochs=spec.epochs), indent=2, default=float) + "\n")
    loss_type = ladder_varix_loss(annealer, folder, VALID_IDS, POSITIONS, on_measure=ladder_live_plot(spec, annealer))
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=SNAPSHOT_EVERY, downstream=False)
    pd.Series(annealer.recon, name="recon_valid").rename_axis("epoch").to_csv(folder / "ladder_recon.csv")
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    LiveDashboard(folder).update_metrics()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, train_seconds=trained,
                                                     total_seconds=time.time() - started), default=float) + "\n")


if RUN_LADDER_TRAINING:
    for spec in ladder_grid:
        if not (RUNS / spec.name / "run.json").exists():
            print(f"{time.strftime('%H:%M')} Lauf {spec.name}", flush=True)
            train_ladder(spec)
ladder_done = [spec for spec in ladder_grid if (RUNS / spec.name / "run.json").exists()]
print("fertig:", len(ladder_done), "von", len(ladder_grid))

_Noch keine Messung._

fertig: 3 von 3


### 11.1 Inseln: welche Krebsart-Gruppen überleben welche Stufe?

**Warum kein Cluster-Count.** HDBSCAN auf z liefert im Arbeitsbereich meist 0 Cluster (eine Wolke ohne Dichtetal,
`trustworthiness_09` Abschnitt 3.4), der Count springt also von „viele“ auf „keine“ und sagt nichts über den Weg dazwischen.
Deshalb werden die Inseln einmal festgelegt und danach nur noch verfolgt.

**Referenz-Inseln.** HDBSCAN (`min_cluster_size = 10`, `min_samples = 5`, `allow_single_cluster = False`) auf z am Ende
der β = 0-Stufe (letzter Snapshot vor dem ersten Stufenwechsel), auf den 800 Controller-Patienten. Jede Insel bekommt
ihre häufigste Krebsart und deren Anteil (Reinheit) als Name. Rauschpunkte gehören zu keiner Insel.

**Kohäsion einer Insel I zu jeder späteren Epoche.** Für jeden Insel-Patienten die k = 20 nächsten Nachbarn in z (alle
800 Patienten, euklidisch); `share` = Anteil der Nachbarn, die in derselben Referenz-Insel liegen. Normiert auf
Zufall = 0 und bestmöglich = 1:

  Kohäsion = (share − chance) / (top − chance), chance = (n_I − 1)/(n − 1), top = min(1, (n_I − 1)/k).

`top` ist nötig, weil eine Insel mit 8 Mitgliedern höchstens 7 von 20 Nachbarn stellen kann. Gemittelt über das
gespeicherte z und 2 neue Ziehungen aus μ, σ (RNG-Seed 0). **Aufgelöst** heißt: erste Epoche nach der Referenz mit
Kohäsion < 0,5 (willkürliche, feste Schwelle). Sensitivität: dieselbe Rechnung mit `min_cluster_size` 5 und 20.

In [8]:
from sklearn.cluster import HDBSCAN
from sklearn.neighbors import NearestNeighbors

ISLAND = dict(min_cluster_size=10, min_samples=5, k=20, draws=2, dissolved=0.5)
ISLAND_SIZES = (5, 10, 20)
LADDER_ROOT = OUTPUT / "ladder"
LADDER_ROOT.mkdir(exist_ok=True)
ladder_names = {spec.seed: spec.name for spec in ladder_done}


def ladder_levels(log):
    """Stufe je Epoche und letzte Snapshot-Epoche jeder Stufe (= Plateau-Zustand)."""
    level = log.set_index("epoch").state.str.removeprefix("level_").astype(int)
    ends = level.groupby(level).apply(lambda part: part.index.max() // SNAPSHOT_EVERY * SNAPSHOT_EVERY)
    return level, ends


def reference_subset(folder, epoch):
    """μ, logvar, z eines Snapshots auf den 800 Controller-Patienten (feste Reihenfolge)."""
    with np.load(folder / "controller_validation_reference.npz") as reference:
        ids = reference["sample_ids"].astype(str)
    snap = load_snapshot(folder, epoch, "valid")
    positions = pd.Index(snap["sample_ids"]).get_indexer(ids)
    return ids, {key: snap[key][positions] for key in ("mu", "logvar", "z")}


def cohesion(z, islands, k=ISLAND["k"]):
    """Normierte kNN-Kohäsion je Referenz-Insel (0 = Zufall, 1 = bestmöglich)."""
    neighbours = NearestNeighbors(n_neighbors=k + 1).fit(z).kneighbors(z, return_distance=False)[:, 1:]
    n, values = len(z), {}
    for island in sorted(set(islands) - {-1}):
        members = islands == island
        size = members.sum()
        chance, top = (size - 1) / (n - 1), min(1.0, (size - 1) / k)
        values[island] = ((islands[neighbours[members]] == island).mean() - chance) / (top - chance)
    return pd.Series(values)


def z_draws(state, draws=ISLAND["draws"], seed=0):
    rng = np.random.default_rng(seed)
    sigma = np.exp(state["logvar"] / 2)
    return [state["z"]] + [state["mu"] + sigma * rng.standard_normal(state["mu"].shape) for _ in range(draws)]


def track_islands(seed):
    folder = RUNS / ladder_names[seed]
    cache = LADDER_ROOT / f"islands_s{seed}.parquet"
    log = pd.read_csv(folder / "controller_log.csv")
    level, ends = ladder_levels(log)
    reference_epoch = int(ends[0])
    if cache.exists():
        return pd.read_parquet(cache), reference_epoch
    ids, state = reference_subset(folder, reference_epoch)
    cancer = labels_of(folder, "valid").loc[ids, "CANCER_TYPE_ACRONYM"].to_numpy()
    epochs = range(reference_epoch, int(log.epoch.max()) + 1, SNAPSHOT_EVERY)
    states = {epoch: reference_subset(folder, epoch)[1] for epoch in epochs}
    rows = []
    for size in ISLAND_SIZES:
        islands = HDBSCAN(min_cluster_size=size, min_samples=ISLAND["min_samples"]).fit_predict(state["z"])
        names = {}
        for island in sorted(set(islands) - {-1}):
            counts = pd.Series(cancer[islands == island]).value_counts()
            names[island] = (counts.index[0], counts.iloc[0] / counts.sum(), int(counts.sum()))
        for epoch, current in states.items():
            values = pd.concat([cohesion(z, islands) for z in z_draws(current)], axis=1).mean(axis=1)
            for island, value in values.items():
                rows.append(dict(seed=seed, min_cluster_size=size, epoch=epoch, level=int(level[epoch]),
                                 beta_over_d=LADDER_LEVELS[level[epoch]], island=island, cancer=names[island][0],
                                 purity=names[island][1], n=names[island][2], cohesion=value,
                                 noise=float((islands == -1).mean())))
    table = pd.DataFrame(rows)
    table.to_parquet(cache)
    return table, reference_epoch


islands, island_reference = {}, {}
for seed in ladder_names:
    islands[seed], island_reference[seed] = track_islands(seed)
islands = pd.concat(islands.values(), ignore_index=True)

overview = (islands[islands.epoch.eq(islands.seed.map(island_reference))]
            .groupby(["seed", "min_cluster_size"])
            .agg(islands=("island", "nunique"), noise=("noise", "first"), purity_median=("purity", "median"),
                 cancer_types=("cancer", "nunique")))
display(overview.round(2))

islands  noise  purity_median  cancer_types
seed min_cluster_size                                             
17   5                      16   0.36           1.00            13
     10                      9   0.36           0.90             9
     20                      5   0.38           0.90             5
23   5                      25   0.47           1.00            17
     10                      3   0.09           1.00             3
     20                      2   0.11           0.56             2
31   5                       2   0.01           0.55             2
     10                     13   0.48           1.00            11
     20                      2   0.13           0.53             2

In [9]:
main = islands[islands.min_cluster_size.eq(ISLAND["min_cluster_size"])]
seeds = sorted(ladder_names)
fig, axes = plt.subplots(2, len(seeds), figsize=(7 * len(seeds), 8.5), sharex=True, sharey="row")
for column, seed in enumerate(seeds):
    part = main[main.seed.eq(seed)]
    ax = axes[0, column]
    for island, line in part.groupby("island"):
        line = line.sort_values("epoch")
        ax.plot(line.epoch, line.cohesion.rolling(3, min_periods=1, center=True).mean(), lw=1.3,
                label=f"{line.cancer.iloc[0]} ({line.purity.iloc[0]:.0%}, n = {line.n.iloc[0]})")
    ax.axhline(ISLAND["dissolved"], color="k", ls=":", lw=1)
    ax.set_title(f"{ladder_names[seed]}: {part.island.nunique()} Inseln bei Epoche {island_reference[seed]}")
    ax.legend(fontsize=7, ncol=2)
    log = pd.read_csv(RUNS / ladder_names[seed] / "controller_log.csv")
    level, ends = ladder_levels(log)
    for ax in axes[:, column]:
        for start in level.index[level.ne(level.shift())][1:]:
            ax.axvline(start, color="grey", lw=.6)
    for size, color in zip(ISLAND_SIZES, ("#7f8c8d", "#c0392b", "#2c3e50")):
        mean = islands[islands.seed.eq(seed) & islands.min_cluster_size.eq(size)].groupby("epoch").cohesion.mean()
        axes[1, column].plot(mean.index, mean.values, color=color, label=f"min_cluster_size {size}")
    axes[1, column].axhline(ISLAND["dissolved"], color="k", ls=":", lw=1)
    axes[1, column].set_xlabel("Epoche (grau: Stufenwechsel)")
    axes[1, column].set_xscale("symlog", linthresh=1000)
axes[0, 0].set_ylabel("Kohäsion je Insel (gleitend über 3 Snapshots)")
axes[1, 0].set_ylabel("mittlere Kohäsion")
axes[1, 0].legend(fontsize=8)
fig.suptitle("Referenz-Inseln aus z am Ende von β = 0, verfolgt über die Leiter (gepunktet: Schwelle 0,5)", fontsize=11)
fig.tight_layout(); fig.savefig(LADDER_ROOT / "island_cohesion.png", dpi=160); plt.show()

In [10]:
dissolve_rows = []
for (seed, size, island), line in islands.groupby(["seed", "min_cluster_size", "island"]):
    line = line.sort_values("epoch")
    after = line[line.epoch.gt(island_reference[seed]) & line.cohesion.lt(ISLAND["dissolved"])]
    first = after.iloc[0] if len(after) else None
    dissolve_rows.append(dict(seed=seed, min_cluster_size=size, island=island, cancer=line.cancer.iloc[0],
                              n=line.n.iloc[0], purity=line.purity.iloc[0],
                              dissolved_epoch=first.epoch if first is not None else np.nan,
                              dissolved_beta_over_d=first.beta_over_d if first is not None else np.nan))
dissolve = pd.DataFrame(dissolve_rows)
dissolve.to_csv(LADDER_ROOT / "island_dissolve.csv", index=False)
display(dissolve[dissolve.min_cluster_size.eq(ISLAND["min_cluster_size"])]
        .sort_values(["seed", "dissolved_epoch"]).set_index(["seed", "island"])
        .style.format(dict(purity="{:.2f}", dissolved_epoch="{:.0f}", dissolved_beta_over_d="{:.2e}")))
by_level = (dissolve.assign(beta=dissolve.dissolved_beta_over_d.fillna(np.inf))
            .groupby(["min_cluster_size", "seed"]).beta.value_counts().unstack(fill_value=0)
            .rename(columns=lambda b: "nie" if np.isinf(b) else f"β/d = {b:g}"))
display(Markdown("**Anzahl Inseln, die auf der jeweiligen Stufe zuerst unter 0,5 fallen** (Sensitivität über `min_cluster_size`)"))
display(by_level)

**Anzahl Inseln, die auf der jeweiligen Stufe zuerst unter 0,5 fallen** (Sensitivität über `min_cluster_size`)

beta                   β/d = 1.25e-05  β/d = 5e-05  β/d = 0.0001  \
min_cluster_size seed                                              
5                17                 0            0             5   
                 23                 0            7             6   
                 31                 0            0             0   
10               17                 0            0             4   
                 23                 0            0             0   
                 31                 2            2             1   
20               17                 0            0             1   
                 23                 0            0             0   
                 31                 0            0             0   

beta                   β/d = 0.00025  β/d = 0.0005  β/d = 0.001  β/d = 0.002  \
min_cluster_size seed                                                          
5                17                6             2            3            0   
                 23                9             2            1            0   
                 31                1             0            0            0   
10               17                1             2            2            0   
                 23                2             0            1            0   
                 31                4             2            1            1   
20               17                1             2            1            0   
                 23                1             0            1            0   
                 31                0             0            1            1   

beta                   β/d = 0.004  
min_cluster_size seed               
5                17              0  
                 23              0  
                 31              1  
10               17              0  
                 23              0  
                 31              0  
20               17              0  
                 23              0  
                 31              0

### 11.2 Was passiert beim Auflösen: Verschmelzen oder Zerfließen?

Für jede Insel I am Plateau-Ende jeder Stufe: Anteil der k = 20 Nachbarn ihrer Patienten, die in Insel J liegen
(Zeile normiert, „Rauschen“ = Nachbar gehört zu keiner Insel). Hohe Werte außerhalb der Diagonale heißen: I verschmilzt
mit J; ein gleichmäßig verteilter Rest heißt: I zerfließt in die ganze Wolke. Dazu je Stufe Rate, kollabierte Dimensionen
(ε-δ-Kriterium wie oben), Anteil μ = 0 je Dimension (Varix-Klemme) und Krebs-/Geschlechts-AUC auf z, neben den Werten von
Folie 42 (dort je β ein eigenes Modell, 600 Epochen ab Zufall; Werte von der Folie abgelesen).

In [11]:
SLIDE_42 = pd.DataFrame(dict(beta_over_d=[0, 1.25e-5, 5e-5, 1e-4, 2.5e-4, 1e-3, 4e-3],
                             auc_cancer_slide42=[.97, .96, .97, .96, .95, .89, .68],
                             auc_sex_slide42=[.97, .86, .72, .73, .66, .63, .54]))


def merge_matrix(z, islands, k=ISLAND["k"]):
    neighbours = NearestNeighbors(n_neighbors=k + 1).fit(z).kneighbors(z, return_distance=False)[:, 1:]
    keep = islands != -1
    frame = pd.DataFrame(dict(source=np.repeat(islands[keep], k), target=islands[neighbours[keep]].ravel()))
    matrix = pd.crosstab(frame.source, frame.target, normalize="index")
    return matrix.rename(columns={-1: "Rauschen"})


def level_table(seed):
    cache = LADDER_ROOT / f"levels_s{seed}.csv"
    if cache.exists():
        return pd.read_csv(cache)
    folder = RUNS / ladder_names[seed]
    log = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    _, ends = ladder_levels(log.reset_index())
    rows = []
    for stage, epoch in ends.items():
        epoch = int(epoch)
        mu, logvar = (load_snapshot(folder, epoch, "valid")[key] for key in ("mu", "logvar"))
        kl = 0.5 * (mu ** 2 + np.exp(logvar) - 1 - logvar)
        collapsed = np.flatnonzero(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)
        zero = (mu == 0).mean(axis=0)
        rows.append(dict(seed=seed, level=stage, beta_over_d=LADDER_LEVELS[stage], epoch=epoch,
                         rate=float(kl.sum(1).mean()), collapsed=" ".join(map(str, collapsed)) or "–",
                         mu_zero_max=f"{zero.max():.2f} (Dim. {zero.argmax()})",
                         trustworthiness=log.metric_value.loc[:epoch].dropna().iloc[-1],
                         **auc_on_z(folder, epoch)))
    table = pd.DataFrame(rows)
    table.to_csv(cache, index=False)
    return table


levels = pd.concat([level_table(seed) for seed in seeds], ignore_index=True)
levels = levels.merge(SLIDE_42, on="beta_over_d", how="left")
display(levels.set_index(["seed", "level"]).style.format(precision=3).format(dict(beta_over_d="{:.2e}", rate="{:.1f}")))

In [12]:
MERGE_LEVELS = (4, 6, 8)  # β/d = 2,5·10⁻⁴, 10⁻³, 4·10⁻³
fig, axes = plt.subplots(len(seeds), len(MERGE_LEVELS), figsize=(6 * len(MERGE_LEVELS), 5.2 * len(seeds)))
for row, seed in enumerate(seeds):
    folder = RUNS / ladder_names[seed]
    ids, reference = reference_subset(folder, island_reference[seed])
    reference_islands = HDBSCAN(min_cluster_size=ISLAND["min_cluster_size"],
                                min_samples=ISLAND["min_samples"]).fit_predict(reference["z"])
    names = (main[main.seed.eq(seed)].drop_duplicates("island").set_index("island").cancer)
    ends = levels[levels.seed.eq(seed)].set_index("level").epoch
    for column, stage in enumerate(MERGE_LEVELS):
        z = reference_subset(folder, int(ends[stage]))[1]["z"]
        matrix = merge_matrix(z, reference_islands)
        labels = [f"{i}:{names[i]}" if i in names else str(i) for i in matrix.columns]
        ax = axes[row, column]
        ax.imshow(matrix.to_numpy(), vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(range(matrix.shape[1]), labels, rotation=90, fontsize=6)
        ax.set_yticks(range(matrix.shape[0]), [f"{i}:{names[i]}" for i in matrix.index], fontsize=6)
        ax.set_title(f"s{seed}, β/d = {LADDER_LEVELS[stage]:g} (Epoche {ends[stage]})", fontsize=9)
fig.suptitle("Wohin zeigen die Nachbarn einer Insel? (Zeile = Insel, Spalte = Insel der Nachbarn)", fontsize=11)
fig.tight_layout(); fig.savefig(LADDER_ROOT / "island_merge.png", dpi=160); plt.show()

## 12. Konvergierte Referenzmodelle: β/d = 0 und β/d = 10⁻³ von Grund auf

**Warum.** Die Plateau-Ende-Zustände der Leiter sind nur in der Rekonstruktion konvergiert; die Rate driftet dort noch
um −2 bis −4 % je 50 Epochen (bei β = 0 sogar +13 bis +46 %), und β/d = 10⁻³ ist ein Pfadzustand nach 100 Epochen ab dem
5·10⁻⁴-Modell. Hier deshalb je ein eigenes Modell pro β und Seed, β konstant ab Epoche 1 (kein Warm-up), 1.000 Epochen,
Snapshots alle 20 Epochen, sonst alles wie Block A.

**Konvergiert** heißt hinterher (die Auswertung liegt in `tomato_ladder.ipynb`): erster Snapshot ab Epoche 600, an dem sich
Validierungs-Rekonstruktion **und** Rate je um höchstens 1 % ändern (Median der letzten 50 Epochen gegen die 50 davor).
AUTOENCODIX trainiert eine feste Epochenzahl; das Kriterium wählt also den Snapshot, es bricht nicht ab.
Laufnamen: `R_conv_bd0_s{seed}` und `R_conv_bd1e-03_s{seed}`. Ein Lauf dauert etwa 30 Minuten.

In [7]:
CONVERGED_BETAS = (0.0, 1e-3)
CONVERGED_EPOCHS = 1000
RUN_CONVERGED_TRAINING = True


class ConstantAnnealer(MaximinAnnealer):
    """Konstantes β ab Epoche 1; schreibt die Validierungs-Rekonstruktion mit und misst Trustworthiness nur mit."""

    def __init__(self, constraints, every=SNAPSHOT_EVERY):
        super().__init__(constraints, warmup_epochs=every, every=every)
        self.w, self.state, self.reason, self.recon = 1.0, "constant", "", {}

    def record(self, epoch, recon):
        self.recon[epoch] = recon

    def update(self, margins):
        return min(margins.values())


@dataclass(frozen=True)
class ConvergedSpec:
    beta_over_d: float
    seed: int
    epochs: int = CONVERGED_EPOCHS

    @property
    def name(self):
        return f"R_conv_bd{'0' if self.beta_over_d == 0 else f'{self.beta_over_d:.0e}'}_s{self.seed}"


converged_grid = [ConvergedSpec(beta, seed) for seed in SEEDS for beta in CONVERGED_BETAS]


def train_converged(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    annealer = ConstantAnnealer([trustworthiness_constraint(0.9, 20)])
    config = build_varix_config(data_config, "tcga", dict(TRAINING, epochs=spec.epochs),
                                dict(RUN_DEFAULTS, beta=spec.beta_over_d * LATENT_DIM), global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, annealer="ConstantAnnealer", n_subset=N_SUBSET, subset_seed=SUBSET_SEED,
        epochs=spec.epochs), indent=2, default=float) + "\n")
    loss_type = ladder_varix_loss(annealer, folder, VALID_IDS, POSITIONS)
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=SNAPSHOT_EVERY, downstream=False)
    pd.Series(annealer.recon, name="recon_valid").rename_axis("epoch").to_csv(folder / "ladder_recon.csv")
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, train_seconds=trained,
                                                     total_seconds=time.time() - started), default=float) + "\n")


if RUN_CONVERGED_TRAINING:
    for spec in converged_grid:
        if not (RUNS / spec.name / "run.json").exists():
            print(f"{time.strftime('%H:%M')} Lauf {spec.name}", flush=True)
            train_converged(spec)
converged_done = [spec for spec in converged_grid if (RUNS / spec.name / "run.json").exists()]
print("fertig:", len(converged_done), "von", len(converged_grid), [spec.name for spec in converged_done])

14:05 Lauf R_conv_bd0_s17
Reproducibility settings for device cuda are not implemented or necessary i.e. for cpu.


You are using a CUDA device ('NVIDIA GeForce RTX 4070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Epoch 1 - Train Loss: 0.0022
Sub-losses: recon_loss: 0.0022, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 1 - Valid Loss: 0.0010
Sub-losses: recon_loss: 0.0010, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 2 - Train Loss: 0.0011
Sub-losses: recon_loss: 0.0011, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 2 - Valid Loss: 0.0009
Sub-losses: recon_loss: 0.0009, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 3 - Train Loss: 0.0009
Sub-losses: recon_loss: 0.0009, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 3 - Valid Loss: 0.0009
Sub-losses: recon_loss: 0.0009, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 4 - Train Loss: 0.0008
Sub-losses: recon_loss: 0.0008, var_loss: 0.0000, anneal_factor: 1.0000, effective_beta_factor: 0.0000
Epoch 4 - Valid Loss: 0.0008
Sub-losses: recon_loss: 0.0008, var_loss: 0.000

## 13. Umschaltversuch: erst T = 0,75 (Collapse), dann T = 0,9 — ist der Collapse umkehrbar?

**Frage.** In Block A kollabieren bei Ziel T(μ) = 0,75 alle drei Seeds. Kommt das Modell zurück, wenn man das Ziel danach
auf 0,9 stellt? Die Antwort entscheidet das Controller-Design: Ist Collapse umkehrbar, darf ein Annealer über die Kante
schießen und zurückregeln; ist er es nicht, muss er vorher anhalten.

**Ablauf** (je Seed 17, 23, 31; sonst alles wie Block A):

| Epochen | β | Sollwert T(μ), k = 20 |
|---|---|---|
| 1–100 | 0 (Warm-up) | – |
| 101–1.100 | Controller | 0,75 |
| 1.101–2.100 | Controller | 0,90 |

Beim Umschalten setzt der Annealer seine Schrittweite auf den Startwert (Faktor 1,25) zurück, damit er nicht mit einer
seit dem Collapse halbierten Schrittweite startet; β selbst wird nicht zurückgesetzt. Der Controller regelt weiter auf
**T(μ)**, wie Block A, damit Phase 1 bis Epoche 595 mit `A_T0.750` identisch ist (deterministisches Training, wird
geprüft). Mitgemessen, ohne zu steuern: E[T(z)], Rate pro Dimension, Anteil μ = 0, Gini der μ-Matrix, AUC auf z.

**Vergleich:** `H_T0.900_k20_e2000` (gleicher Seed, direkt auf 0,9, 2.000 Epochen). Die Frage ist, ob der Endzustand des
Umschaltlaufs (Epoche 2.100) dort ankommt, wo der direkte Lauf steht.

**Hinweis zur Klemme** (hergeleitet aus `_varix_architecture.py`): Sitzt μ für alle Patienten einer Dimension auf 0, hat
diese Dimension über `torch.where` keinen Gradienten aus der Rekonstruktion mehr; zurückkommen kann sie nur, wenn sich die
gemeinsamen Encoder-Gewichte so ändern, dass μ für einzelne Patienten wieder über 10⁻⁷ steigt. In Block A wurden 3 von 47
toten Dimensionen wieder lebendig.

In [6]:
from dataclasses import replace

SWITCH_SCHEDULE = ((0, 0.75), (1100, 0.90))  # (ab abgeschlossener Epoche >, Sollwert)
SWITCH_EPOCHS, SWITCH_K = 2100, 20
RUN_SWITCH_TRAINING = True


class SwitchingMaximinAnnealer(MaximinAnnealer):
    """MaximinAnnealer, dessen Sollwert nach festen Epochen wechselt; beim Wechsel Schrittweite auf den Startwert."""

    def __init__(self, constraints, schedule, **kwargs):
        super().__init__(constraints, **kwargs)
        self.schedule, self.initial_step = tuple(schedule), self.step

    def target_at(self, done):
        return [target for start, target in self.schedule if done > start][-1]

    def measures_at(self, epoch, total_epochs):
        target = self.target_at(epoch + 1)
        if target != self.constraints[0].target:
            self.constraints = (replace(self.constraints[0], target=target),) + self.constraints[1:]
            self.step, self.direction = self.initial_step, 0
            self.state, self.reason = "seeking", "target_switch"
        return super().measures_at(epoch, total_epochs)


@dataclass(frozen=True)
class SwitchSpec:
    seed: int
    k: int = SWITCH_K
    epochs: int = SWITCH_EPOCHS

    @property
    def name(self):
        first, second = (target for _, target in SWITCH_SCHEDULE)
        return f"S_T{first:.3f}to{second:.3f}_k{self.k}_e{self.epochs}_s{self.seed}"


# Test: Ziel wechselt genau nach Epoche 1.100, Schrittweite wird zurückgesetzt, β nicht.
probe = SwitchingMaximinAnnealer([Constraint("t", None, .75, .01)], SWITCH_SCHEDULE, **CONTROLLER)
probe.w, probe.step, probe.direction = 0.3, probe.min_step, -1
for epoch in (99, 1098, 1099):  # 0-basiert: abgeschlossene Epochen 100, 1.099, 1.100
    probe.measures_at(epoch, SWITCH_EPOCHS)
assert probe.constraints[0].target == .75 and probe.step == probe.min_step
probe.measures_at(1100, SWITCH_EPOCHS)  # abgeschlossene Epoche 1.101
assert probe.constraints[0].target == .90 and probe.step == probe.initial_step and probe.direction == 0
assert probe.w == 0.3 and probe.reason == "target_switch"
print("Umschalten geprüft: Ziel 0,75 bis Epoche 1.100, ab 1.101 Ziel 0,90; Schrittweite zurückgesetzt, β unverändert.")

switch_grid = [SwitchSpec(seed) for seed in SEEDS]
switch_handle = display(Markdown("_Noch keine Messung._"), display_id=True)


def switch_live_plot(spec):
    def draw(log):
        measured = log.dropna(subset=["metric_value"])
        fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", color="black")
        axes[0].set(yscale="log", title="β/d", xlabel="Epoche")
        axes[1].plot(measured.epoch, measured.metric_value, "o-", ms=2)
        axes[1].plot(measured.epoch, measured.target, color="grey", lw=.8)
        axes[1].set(title=f"Trustworthiness auf μ (k = {spec.k})", xlabel="Epoche")
        axes[2].plot(log.epoch, log.rate, lw=1.2)
        axes[2].set(yscale="log", title="Rate", xlabel="Epoche")
        axes[3].plot(log.epoch, log.collapsed_dimensions, lw=1.2)
        axes[3].set(title="kollabierte Dimensionen (ε-δ)", xlabel="Epoche")
        for axis in axes:
            axis.axvline(SWITCH_SCHEDULE[1][0], color="grey", ls="--", lw=.8)
        fig.suptitle(spec.name)
        fig.tight_layout()
        switch_handle.update(fig)
        plt.close(fig)
    return draw


def train_switch(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    annealer = SwitchingMaximinAnnealer([trustworthiness_constraint(SWITCH_SCHEDULE[0][1], spec.k)], SWITCH_SCHEDULE,
                                        **CONTROLLER)
    config = build_varix_config(data_config, "tcga", dict(TRAINING, epochs=spec.epochs),
                                dict(RUN_DEFAULTS, beta=BETA_OVER_D_MAX * LATENT_DIM), global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, annealer="SwitchingMaximinAnnealer", schedule=SWITCH_SCHEDULE, beta_over_d_max=BETA_OVER_D_MAX,
        controller=CONTROLLER, n_subset=N_SUBSET, subset_seed=SUBSET_SEED, epochs=spec.epochs), indent=2, default=float) + "\n")
    loss_type = annealed_varix_loss(annealer, folder, VALID_IDS, POSITIONS, on_measure=switch_live_plot(spec))
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=SNAPSHOT_EVERY, downstream=False)
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    LiveDashboard(folder).update_metrics()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, train_seconds=trained,
                                                     total_seconds=time.time() - started), default=float) + "\n")


if RUN_SWITCH_TRAINING:
    for spec in switch_grid:
        if not (RUNS / spec.name / "run.json").exists():
            print(f"{time.strftime('%H:%M')} Lauf {spec.name}", flush=True)
            train_switch(spec)
switch_done = [spec for spec in switch_grid if (RUNS / spec.name / "run.json").exists()]
print("fertig:", len(switch_done), "von", len(switch_grid))

Umschalten geprüft: Ziel 0,75 bis Epoche 1.100, ab 1.101 Ziel 0,90; Schrittweite zurückgesetzt, β unverändert.


_Noch keine Messung._

fertig: 3 von 3


### 13.1 Auswertung

Aus dem Controller-Log (jede Epoche): β/d, T(μ), Rate, kollabierte Dimensionen. Aus den Snapshots (alle 20 Epochen):
Rate R_j pro Dimension, Anteil μ = 0, Gini der ganzen μ-Matrix (Hurley & Rickard 2009, wie `gini_index`); alle 100 Epochen
zusätzlich E[T(z)] (2 Ziehungen) und AUC auf z. Vergleich mit `H_T0.900_k20_e2000` (direkt auf 0,9) an denselben Epochen.

**Zurückgekommen** heißt für eine Dimension: am Ende von Phase 1 nach ε-δ kollabiert, am Laufende nicht mehr.

In [7]:
SWITCH_AT = SWITCH_SCHEDULE[1][0]
DIRECT_EPOCHS = 2000


def direct_name(seed):
    """Vergleichslauf direkt auf 0,9 aus Abschnitt 10.2 (Arm H)."""
    return f"H_T0.900_k20_e{DIRECT_EPOCHS}_s{seed}"


def gini_all(mu):
    values = np.sort(np.abs(np.ravel(mu)))
    total, n = values.sum(), len(values)
    k = np.arange(1, n + 1)
    return float(1 - 2 * ((values / total) * (n - k + 0.5) / n).sum()) if total > 0 else np.nan


def analyse_switch(name, epochs):
    folder = RUNS / name
    cache = folder / "analysis_switch.csv"
    if cache.exists():
        return pd.read_csv(cache)
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    log = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    rows = []
    for epoch in range(SNAPSHOT_EVERY, epochs + 1, SNAPSHOT_EVERY):
        valid = load_snapshot(folder, epoch, "valid")
        mu, logvar = valid["mu"].astype(np.float64), valid["logvar"].astype(np.float64)
        kl = compute_conditional_kl_matrix(mu, logvar)
        collapsed = np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta
        row = dict(epoch=epoch, beta_over_d=log.loc[epoch, "beta_over_d"], rate=float(kl.sum(1).mean()),
                   collapsed_dimensions=int(collapsed.sum()), collapsed_set=",".join(map(str, np.flatnonzero(collapsed))),
                   mu_zero_share=float((mu == 0).mean()), gini_mu=gini_all(mu),
                   **{f"rate_dim{j}": float(value) for j, value in enumerate(kl.mean(0))})
        if epoch % 100 == 0 or epoch in (SWITCH_AT, epochs):
            positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
            subset, sigma = mu[positions], np.exp(0.5 * logvar[positions])
            rng = np.random.default_rng(SUBSET_SEED)
            row |= dict(t_mu=float(trustworthiness(x, subset, n_neighbors=20)) if len(np.unique(subset, axis=0)) > 20 else np.nan,
                        t_z=float(np.mean([trustworthiness(x, subset + sigma * rng.standard_normal(subset.shape), n_neighbors=20)
                                           for _ in range(2)])),
                        **auc_on_z(folder, epoch))
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(cache, index=False)
    return table


switch = {spec.seed: analyse_switch(spec.name, spec.epochs) for spec in switch_done}
direct = {seed: analyse_switch(direct_name(seed), DIRECT_EPOCHS) for seed in switch
          if (RUNS / direct_name(seed) / "run.json").exists()}

# Determinismus: Phase 1 bis Epoche 595 identisch mit A_T0.750 (gleicher Seed, gleicher Sollwert)
for spec in switch_done:
    reference = RUNS / RunSpec("A", spec.seed, 0.75).name / "controller_log.csv"
    if reference.exists():
        a = pd.read_csv(reference).set_index("epoch").effective_beta.loc[:595]
        s = pd.read_csv(RUNS / spec.name / "controller_log.csv").set_index("epoch").effective_beta.loc[:595]
        print(f"{spec.name}: β bis Epoche 595 identisch mit A_T0.750: {np.allclose(a, s, rtol=1e-12, atol=0)}")

S_T0.750to0.900_k20_e2100_s17: β bis Epoche 595 identisch mit A_T0.750: True
S_T0.750to0.900_k20_e2100_s23: β bis Epoche 595 identisch mit A_T0.750: True
S_T0.750to0.900_k20_e2100_s31: β bis Epoche 595 identisch mit A_T0.750: True


In [8]:
fig, axes = plt.subplots(2, 4, figsize=(21, 8.5))
seed_colors = dict(zip(SEEDS, ("tab:blue", "tab:orange", "tab:green")))
for spec in switch_done:
    log = pd.read_csv(RUNS / spec.name / "controller_log.csv")
    measured = log.dropna(subset=["metric_value"])
    table = switch[spec.seed]
    style = dict(color=seed_colors[spec.seed], lw=1.1, label=f"0,75 → 0,9, s{spec.seed}")
    axes[0, 0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", **style)
    axes[0, 1].plot(measured.epoch, measured.metric_value, **style)
    axes[0, 2].plot(log.epoch, log.rate, **style)
    axes[0, 3].plot(log.epoch, log.collapsed_dimensions, **style)
    sparse = table.dropna(subset=["t_z"])
    axes[1, 0].plot(sparse.epoch, sparse.t_z, "o-", ms=3, **style)
    axes[1, 1].plot(sparse.epoch, sparse.auc_cancer, "o-", ms=3, **style)
    axes[1, 2].plot(table.epoch, table.gini_mu, **style)
    axes[1, 3].plot(table.epoch, table.mu_zero_share, **style)
    if spec.seed in direct:
        reference = direct[spec.seed]
        ref_log = pd.read_csv(RUNS / direct_name(spec.seed) / "controller_log.csv")
        ref_measured = ref_log.dropna(subset=["metric_value"])
        dashed = dict(color=seed_colors[spec.seed], lw=.9, ls=":", label=f"direkt 0,9, s{spec.seed}")
        axes[0, 0].step(ref_log.epoch, ref_log.beta_over_d.clip(lower=1e-9), where="post", **dashed)
        axes[0, 1].plot(ref_measured.epoch, ref_measured.metric_value, **dashed)
        axes[0, 2].plot(ref_log.epoch, ref_log.rate, **dashed)
        axes[0, 3].plot(ref_log.epoch, ref_log.collapsed_dimensions, **dashed)
        ref_sparse = reference.dropna(subset=["t_z"])
        axes[1, 0].plot(ref_sparse.epoch, ref_sparse.t_z, **dashed)
        axes[1, 1].plot(ref_sparse.epoch, ref_sparse.auc_cancer, **dashed)
        axes[1, 2].plot(reference.epoch, reference.gini_mu, **dashed)
        axes[1, 3].plot(reference.epoch, reference.mu_zero_share, **dashed)
titles = ["β/d", "T(μ), Controller (Sollwert 0,75 → 0,9)", "Rate (Nats)", "kollabierte Dimensionen (ε-δ)",
          "E[T(z)] (2 Ziehungen)", "AUC Krebsart (z)", "Gini ganze μ-Matrix", "Anteil μ = 0"]
for axis, title in zip(axes.ravel(), titles):
    axis.axvline(SWITCH_AT, color="grey", ls="--", lw=.8)
    axis.set(title=title, xlabel="Epoche")
axes[0, 0].set_yscale("log")
axes[0, 2].set_yscale("log")
axes[0, 1].axhspan(.74, .76, xmax=SWITCH_AT / SWITCH_EPOCHS, alpha=.12)
axes[0, 1].axhspan(.89, .91, alpha=.12)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Umschaltversuch: durchgezogen 0,75 → 0,9, gepunktet direkt 0,9 (H); gestrichelt senkrecht das Umschalten")
fig.tight_layout()
fig.savefig(OUTPUT / "switch_075_090.png", dpi=150)
plt.show()

In [9]:
fig, axes = plt.subplots(1, len(switch_done), figsize=(6.5 * max(len(switch_done), 1), 4), squeeze=False)
for axis, spec in zip(axes[0], switch_done):
    table = switch[spec.seed]
    for j in range(LATENT_DIM):
        axis.plot(table.epoch, table[f"rate_dim{j}"].clip(lower=1e-5), color=plt.cm.tab10(j), lw=1, label=f"dim {j}")
    axis.axvline(SWITCH_AT, color="grey", ls="--", lw=.8)
    axis.axhline(COLLAPSE.primary_epsilon, color="k", ls=":", lw=.8)
    axis.set(yscale="log", title=f"s{spec.seed}: Rate pro Dimension", xlabel="Epoche", ylabel="R_j (Nats)")
axes[0, -1].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT / "switch_rate_per_dimension.png", dpi=150)
plt.show()

summary_rows = []
for spec in switch_done:
    table = switch[spec.seed].set_index("epoch")
    end_1, end_2 = table.loc[SWITCH_AT], table.iloc[-1]
    dead_1 = set(filter(None, str(end_1.collapsed_set).split(","))) if isinstance(end_1.collapsed_set, str) else set()
    dead_2 = set(filter(None, str(end_2.collapsed_set).split(","))) if isinstance(end_2.collapsed_set, str) else set()
    for moment, row in (("Ende 0,75 (Ep. 1.100)", end_1), ("Ende 0,9 (Ep. 2.100)", end_2)):
        summary_rows.append(dict(seed=spec.seed, Zustand=moment, beta_over_d=row.beta_over_d, Rate=row.rate,
                                 kollabiert=int(row.collapsed_dimensions), T_mu=row.t_mu, T_z=row.t_z,
                                 AUC_Krebs=row.auc_cancer, AUC_Geschlecht=row.auc_sex_mixed, Gini=row.gini_mu,
                                 Null_Anteil=row.mu_zero_share))
    if spec.seed in direct:
        row = direct[spec.seed].iloc[-1]
        summary_rows.append(dict(seed=spec.seed, Zustand=f"direkt 0,9 (Ep. {int(row.epoch)})", beta_over_d=row.beta_over_d,
                                 Rate=row.rate, kollabiert=int(row.collapsed_dimensions), T_mu=row.t_mu, T_z=row.t_z,
                                 AUC_Krebs=row.auc_cancer, AUC_Geschlecht=row.auc_sex_mixed, Gini=row.gini_mu,
                                 Null_Anteil=row.mu_zero_share))
    summary_rows.append(dict(seed=spec.seed, Zustand="zurückgekommen / neu tot",
                             kollabiert=f"{len(dead_1 - dead_2)} / {len(dead_2 - dead_1)}"))
switch_summary = pd.DataFrame(summary_rows).set_index(["seed", "Zustand"])
switch_summary.to_csv(OUTPUT / "switch_summary.csv")
display(switch_summary.style.format(precision=3, na_rep="").format("{:.2e}", subset=["beta_over_d"], na_rep=""))

### 13.2 Befunde (gemessen, 3 Seeds, TCGA, d = 8)

1. **Phase 1 reproduziert Block A:** β bis Epoche 595 in allen Seeds identisch mit `A_T0.750`. Bei Ziel 0,75 kollabieren
   1 (s23), 4 (s31) und 5 (s17) Dimensionen, alle schon um Epoche 300; β/d am Ende von Phase 1 2,5–2,8·10⁻³.
2. **T(μ) kommt schnell zurück:** Nach dem Umschalten ist T(μ) ab Epoche 1.180–1.195 im Band 0,9 ± 0,01 (80–95 Epochen)
   und danach zu 88–89 % der Messungen im Band; 35–40 β-Änderungen in Phase 2. β fällt dabei kurz auf ≈ 5·10⁻⁵.
3. **Die toten Dimensionen bleiben fast alle tot:** von 10 kollabierten Dimensionen kommen 2 zurück (beide in s17,
   kurz nach dem Umschalten, Epoche ≈ 1.200 und ≈ 1.280); s23 bleibt bei 1, s31 bei 4 toten Dimensionen bis Epoche 2.100,
   obwohl β zwischenzeitlich um zwei Größenordnungen sinkt. **Der Collapse ist bei Varix weitgehend nicht umkehrbar**
   (innerhalb von 1.000 Epochen; kein Beweis permanenter Unumkehrbarkeit).
4. **Hysterese im β:** Um dasselbe T(μ) = 0,9 zu halten, braucht der Umschaltlauf β/d 4,2·10⁻⁴ / 8,2·10⁻⁴ / 4,3·10⁻⁴
   (s17 / s23 / s31), der direkte Lauf 1,08–1,15·10⁻³, also 1,3- bis 2,7-mal weniger β. Die Rate liegt dafür höher
   (4,1–7,1 gegen 3,0–3,5 Nats): die verbliebenen Dimensionen tragen mehr.
5. **Downstream kommt trotzdem zurück, sogar etwas höher:** Epoche 2.100 gegen direkt (Epoche 2.000):
   Krebs-AUC 0,844–0,848 gegen 0,819–0,840, Geschlechts-AUC 0,59–0,61 gegen 0,55–0,57, E[T(z)] 0,71–0,74 gegen
   0,70–0,72. Gini 0,85 gegen 0,88, Null-Anteil 0,78–0,79 gegen 0,79–0,81. Die verlorenen Dimensionen werden also durch
   mehr Rate in den übrigen ausgeglichen; ein Teil des AUC-Vorsprungs kann an der höheren Rate liegen (gleiches T(μ),
   nicht gleiche Rate).
6. **Folgerung für das Controller-Design:** Überschießen über die Kante kostet Dimensionen dauerhaft; T(μ) als Sollwert
   holt die Preservation zurück, nicht aber die Dimensionen. Ein Collapse-Wächter muss deshalb **vor** der Kante
   anhalten (Rate pro Dimension), statt hinterher zurückzuregeln.

Grenzen: ein Datensatz, d = 8, Regelung auf T(μ), 3 Seeds; Collapse nach ε-δ (ε = 0,01 Nats, δ = 1 %).

### 13.3 Wie reagieren die Metriken auf 0,75 und auf 0,9?

Alle Snapshots (alle 20 Epochen) der drei Umschaltläufe und der drei direkten 0,9-Läufe (`H_…`, bis Epoche 2.000), auf
dem festen Controller-Subset (800 Validierungspatienten) bzw. für die AUC auf z (Train → Validierung):

| Gruppe | Metriken |
|---|---|
| Preservation auf μ | T(k = 20), Continuity (k = 20, ZADU), R_NX bei k = 10 / 40 / 200 |
| Preservation auf z | E[T(z)] (2 Ziehungen), R_NX(z) bei k = 10 / 200 |
| Information / Kompression | Rate, Informationsschranke Σ ½ log(1 + SNR_d) (beschreibend), D_eff, tote Dimensionen |
| Sparsity | Gini der ganzen μ-Matrix, Null-Anteil, aktive Dimensionen pro Patient |
| Downstream (nur Bewertung) | AUC Krebsart, AUC Geschlecht in gemischten Krebsarten |

**Erholung** einer Metrik = (Wert Ende 0,9 − Wert Ende 0,75) / (Wert direkt − Wert Ende 0,75): 1 = erreicht den
direkten Lauf, < 1 = bleibt darunter, > 1 = überholt ihn. **Reaktionszeit** = Epochen nach dem Umschalten, bis die
Metrik erstmals 90 % ihrer Gesamtänderung bis Epoche 2.100 erreicht hat.

In [10]:
from zadu.measures import trustworthiness_continuity

REACTION_K = (10, 40, 200)


def reaction_metrics(folder, epoch, x, reference_ids):
    valid = load_snapshot(folder, epoch, "valid")
    mu_all, logvar_all = valid["mu"].astype(np.float64), valid["logvar"].astype(np.float64)
    positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
    mu, sigma = mu_all[positions], np.exp(0.5 * logvar_all[positions])
    kl = compute_conditional_kl_matrix(mu_all, logvar_all)
    rate_dims = kl.mean(0)
    variance = np.exp(logvar_all)
    snr = mu_all.var(0) / variance.mean(0)
    z = mu + sigma * np.random.default_rng(SUBSET_SEED).standard_normal(mu.shape)
    distinct = len(np.unique(mu, axis=0)) > 200
    row = dict(epoch=epoch, rate=float(rate_dims.sum()),
               dead=int(np.sum(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)),
               info_bound=float(0.5 * np.log1p(snr).sum()),
               d_eff=float(rate_dims.sum() ** 2 / (rate_dims ** 2).sum()), gini=gini_all(mu_all),
               zero_share=float((mu_all == 0).mean()), active_per_patient=float((mu_all > 0).sum(1).mean()),
               t_mu=float(trustworthiness(x, mu, n_neighbors=20)) if distinct else np.nan,
               continuity_mu=float(trustworthiness_continuity.measure(x, mu, k=20)["continuity"]) if distinct else np.nan,
               t_z=float(trustworthiness(x, z, n_neighbors=20)),
               **{f"rnx_mu_k{k}": float(r_nx(x, mu, k)) if distinct else np.nan for k in REACTION_K},
               **{f"rnx_z_k{k}": float(r_nx(x, z, k)) for k in (10, 200)})
    return row | auc_on_z(folder, epoch)


def analyse_reaction(name, epochs):
    folder = RUNS / name
    cache = folder / "analysis_reaction.csv"
    if cache.exists():
        return pd.read_csv(cache)
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    table = pd.DataFrame([reaction_metrics(folder, epoch, x, reference_ids)
                          for epoch in range(SNAPSHOT_EVERY, epochs + 1, SNAPSHOT_EVERY)])
    table.to_csv(cache, index=False)
    return table


reaction = {("switch", spec.seed): analyse_reaction(spec.name, spec.epochs) for spec in switch_done}
reaction |= {("direct", seed): analyse_reaction(direct_name(seed), DIRECT_EPOCHS) for seed in direct}
print(f"{len(reaction)} Läufe ausgewertet")

REACTION_PANELS = [
    ("t_mu", "T(μ), k = 20 (Regelgröße)"), ("continuity_mu", "Continuity(μ), k = 20"),
    ("rnx_mu_k10", "R_NX(μ), k = 10"), ("rnx_mu_k40", "R_NX(μ), k = 40"), ("rnx_mu_k200", "R_NX(μ), k = 200"),
    ("t_z", "E[T(z)], k = 20"), ("rnx_z_k10", "R_NX(z), k = 10"), ("rnx_z_k200", "R_NX(z), k = 200"),
    ("rate", "Rate (Nats)"), ("info_bound", "Σ ½ log(1 + SNR)"), ("d_eff", "D_eff"), ("dead", "tote Dimensionen"),
    ("gini", "Gini ganze μ-Matrix"), ("zero_share", "Anteil μ = 0"), ("active_per_patient", "aktive Dims pro Patient"),
    ("auc_cancer", "AUC Krebsart (z)"), ("auc_sex_mixed", "AUC Geschlecht (z)")]

6 Läufe ausgewertet


In [11]:
fig, axes = plt.subplots(5, 4, figsize=(22, 20))
for axis, (column, title) in zip(axes.ravel(), REACTION_PANELS):
    for (arm, seed), table in reaction.items():
        axis.plot(table.epoch, table[column], color=seed_colors[seed], lw=1.1 if arm == "switch" else .9,
                  ls="-" if arm == "switch" else ":", label=f"{'0,75 → 0,9' if arm == 'switch' else 'direkt 0,9'}, s{seed}")
    axis.axvline(CONTROLLER["warmup_epochs"], color="grey", lw=.6)
    axis.axvline(SWITCH_AT, color="grey", ls="--", lw=.8)
    axis.set(title=title, xlabel="Epoche")
    if column in ("rate", "info_bound"):
        axis.set_yscale("log")
for axis in axes.ravel()[len(REACTION_PANELS):]:
    axis.set_axis_off()
axes[0, 0].legend(fontsize=7)
fig.suptitle("Reaktion aller Metriken: durchgezogen 0,75 → 0,9, gepunktet direkt 0,9; grau Ende Warm-up, gestrichelt Umschalten",
             fontsize=12)
fig.tight_layout(rect=(0, 0, 1, .98))
fig.savefig(OUTPUT / "switch_reaction_all.png", dpi=140)
plt.show()

**Normierte Übersicht.** Für jede Metrik und jeden Umschaltlauf: 0 = Wert am Ende des Warm-ups (Epoche 100),
1 = Wert des direkten 0,9-Laufs desselben Seeds am Ende (Epoche 2.000). Median über die drei Seeds, alle 20 Epochen.
So sieht man auf einer Skala, wie weit jede Metrik bei 0,75 über den direkten Zustand hinausschießt (> 1) und ob sie nach
dem Umschalten auf 1 zurückkehrt. Tote Dimensionen sind hier nicht normierbar (direkt = Warm-up = 0) und fehlen.

In [12]:
normalised = []
for (arm, seed), table in reaction.items():
    if arm != "switch" or ("direct", seed) not in reaction:
        continue
    start = table.set_index("epoch").loc[CONTROLLER["warmup_epochs"]]
    end_direct = reaction[("direct", seed)].iloc[-1]
    for column, _ in REACTION_PANELS:
        span = end_direct[column] - start[column]
        if column == "dead" or not np.isfinite(span) or abs(span) < 1e-12:
            continue
        normalised.append(pd.DataFrame(dict(epoch=table.epoch, metric=column, seed=seed,
                                            value=(table[column] - start[column]) / span)))
normalised = pd.concat(normalised).groupby(["metric", "epoch"]).value.median().unstack("epoch")
labels = dict(REACTION_PANELS)
normalised = normalised.loc[[column for column, _ in REACTION_PANELS if column in normalised.index]]
fig, axis = plt.subplots(figsize=(18, 7))
image = axis.imshow(normalised.clip(-1, 3), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=3,
                    extent=(normalised.columns.min() - 10, normalised.columns.max() + 10, len(normalised) - .5, -.5))
axis.set_yticks(range(len(normalised)), [labels[m] for m in normalised.index])
axis.axvline(CONTROLLER["warmup_epochs"], color="k", lw=.6)
axis.axvline(SWITCH_AT, color="k", ls="--", lw=1)
axis.set(xlabel="Epoche", title="Normiert: 0 = Ende Warm-up, 1 = direkter 0,9-Lauf am Ende (Median über 3 Seeds; Farbe auf −1 … 3 begrenzt)")
fig.colorbar(image, ax=axis, shrink=.8, label="normierter Wert")
fig.tight_layout()
fig.savefig(OUTPUT / "switch_reaction_normalised.png", dpi=150)
plt.show()

In [13]:
recovery_rows = []
for column, title in REACTION_PANELS:
    for seed in sorted({seed for arm, seed in reaction if arm == "switch"}):
        if ("direct", seed) not in reaction:
            continue
        table = reaction[("switch", seed)].set_index("epoch")[column]
        end_1, end_2 = table.loc[SWITCH_AT], table.iloc[-1]
        direct_end = reaction[("direct", seed)][column].iloc[-1]
        after = table.loc[SWITCH_AT:]
        change = end_2 - end_1
        reached = after[(after - end_1) / change >= .9].index.min() if abs(change) > 1e-12 else np.nan
        recovery_rows.append(dict(Metrik=title, seed=seed, Ende_075=end_1, Ende_09=end_2, direkt=direct_end,
                                  Erholung=(end_2 - end_1) / (direct_end - end_1) if abs(direct_end - end_1) > 1e-12 else np.nan,
                                  Reaktionszeit=reached - SWITCH_AT if np.isfinite(reached) else np.nan))
recovery = pd.DataFrame(recovery_rows)
recovery.to_csv(OUTPUT / "switch_recovery.csv", index=False)
summary = recovery.groupby("Metrik", sort=False).agg(
    Ende_075=("Ende_075", "median"), Ende_09=("Ende_09", "median"), direkt=("direkt", "median"),
    Erholung=("Erholung", "median"), Erholung_min=("Erholung", "min"), Erholung_max=("Erholung", "max"),
    Reaktionszeit=("Reaktionszeit", "median"))
display(summary.style.format(precision=3).format("{:.0f}", subset=["Reaktionszeit"], na_rep="–")
        .set_caption("Median über 3 Seeds. Erholung 1 = erreicht den direkten Lauf; Reaktionszeit in Epochen nach dem Umschalten (90 % der Änderung)"))

,Ende_075,Ende_09,direkt,Erholung,Erholung_min,Erholung_max,Reaktionszeit
Metrik,,,,,,,
"T(μ), k = 20 (Regelgröße)",0.759,0.907,0.902,1.038,0.999,1.053,100
"Continuity(μ), k = 20",0.836,0.932,0.937,0.968,0.934,0.972,80
"R_NX(μ), k = 10",0.079,0.274,0.269,1.014,1.006,1.049,100
"R_NX(μ), k = 40",0.213,0.439,0.426,1.054,1.033,1.058,100
"R_NX(μ), k = 200",0.321,0.458,0.510,0.729,0.640,0.777,80
"E[T(z)], k = 20",0.576,0.738,0.706,1.233,0.932,1.384,60
"R_NX(z), k = 10",0.009,0.078,0.054,1.522,1.002,1.913,80
"R_NX(z), k = 200",0.062,0.249,0.202,1.331,0.979,1.433,60
Rate (Nats),0.761,6.317,3.498,2.009,1.214,2.812,60


### 13.4 Befunde zur Reaktion der Metriken (gemessen, 3 Seeds, Median; Tabelle oben)

**Hinweis zur Heatmap:** Sie normiert auf (Wert − Warm-up) / (direkt − Warm-up). Wo Warm-up und direkter Lauf fast gleich
sind (R_NX(μ) bei k = 200: 0,50 gegen 0,51; Rate: Tausende Nats gegen 3,5), wird der Nenner winzig und die Farbe
bedeutungslos. Für diese Zeilen gelten nur die Verlaufsbilder und die Tabelle.

**Fahrt auf 0,75 (Epoche ≈ 250–300).** Alle Metriken fallen gleichzeitig, in dem Moment, in dem β kurz über 4·10⁻³ geht und
die Dimensionen sterben. Lokal fällt am stärksten: R_NX(μ) k = 10 von ≈ 0,53 auf 0,08, k = 40 von 0,60 auf 0,21, k = 200
von 0,55 auf 0,32; Continuity(μ) 0,97 → 0,84; E[T(z)] 0,95 → 0,58; R_NX(z) k = 10 0,50 → 0,01. Krebs-AUC 0,98 → 0,65,
Geschlecht 0,9 → 0,51. Sparsity springt: Gini 0,5 → 0,95, Null-Anteil 0,90, 0,8 aktive Dimensionen pro Patient.

**Fahrt auf 0,9 (ab Epoche 1.100).**

1. **Schnell und vollständig zurück:** T(μ) (Erholung 1,04; per Konstruktion), R_NX(μ) bei k = 10 und 40 (1,01 / 1,05),
   Continuity(μ) fast (0,97). Reaktionszeit 80–100 Epochen.
2. **Nicht zurück — die globale Struktur auf μ:** R_NX(μ) bei k = 200 erholt sich nur zu 0,73 (0,46 gegen 0,51 direkt).
   Mit 1–4 toten Dimensionen bleibt die großräumige Ordnung beschädigt, obwohl die lokale wieder stimmt.
3. **Über den direkten Lauf hinaus — alles auf z und die Information:** E[T(z)] 1,23, R_NX(z) 1,3–1,5, SNR-Schranke
   1,25 (2,39 gegen 2,14), Rate 2,0 (6,3 gegen 3,5 Nats), Krebs-AUC 1,08 (0,846 gegen 0,833), Geschlecht 1,6 (0,59 gegen
   0,55; verrauscht). Die übrigen Dimensionen tragen mehr Signal relativ zu σ; deshalb sieht z besser aus als im direkten
   Lauf, obwohl Dimensionen fehlen. Reaktionszeit 60 Epochen.
4. **Code weniger dünn als direkt:** Gini 0,85 gegen 0,88, Null-Anteil 0,78 gegen 0,80, 1,75 gegen 1,58 aktive
   Dimensionen pro Patient (Erholung 1,3–1,5, also über das Ziel hinaus).
5. **D_eff bleibt niedrig** (4,9 gegen 7,4, Erholung 0,46): die Rate verteilt sich auf weniger Dimensionen. Tote
   Dimensionen: 4 → 3 (Median), Erholung 0.
6. **Kurzes Überschwingen direkt nach dem Umschalten** (Epoche ≈ 1.150–1.300): Rate bis ≈ 25 Nats, E[T(z)] bis 0,83,
   Krebs-AUC bis 0,92, aktive Dimensionen pro Patient bis 3,9. Der Controller senkt β zuerst zu weit (≈ 5·10⁻⁵) und
   erhöht es dann wieder; die Metriken folgen β mit etwa 50–100 Epochen Verzögerung.

**Lesart.** Bei gleichem T(μ) ist das „zurückgeholte“ Modell ein anderes: weniger Dimensionen, mehr Rate pro Dimension,
bessere lokale Struktur auf z, aber schlechtere globale Struktur auf μ. T(μ) allein kann das nicht unterscheiden; ein
Wächter auf die Rate pro Dimension und eine globale Metrik (R_NX bei großem k) würden es zeigen.

## 14. Treppe abwärts mit Abzweigen: Wann fallen die Dimensionen, und ab wann gibt es kein Comeback?

**Fragen.** (1) Wie viele Dimensionen leben noch, wenn die Trustworthiness stufenweise immer niedriger angesteuert wird?
(2) Von welchem Zustand aus kommt das Modell zurück auf T(μ) = 0,9, wenn man den Sollwert wieder anhebt? Der tiefste
Zustand, von dem aus das noch gelingt, ist der **Punkt mit der wenigsten Information, der noch ein Comeback erlaubt**;
darunter liegt der point of no return (§13, hergeleitet: mit 0 lebenden Dimensionen bekommt der Encoder keinen Gradienten).

**Treppe:** Warm-up 100 Epochen (β = 0), dann Sollwert T(μ) (k = 20) in Stufen zu **1.000 Epochen** (jede Stufe darf
einschwingen, wie in Block H/F und §13): 0,95 → 0,85 → 0,75 → 0,65 → 0,55 (Ende Epoche 5.100).

**Abzweige:** gleicher Seed, gleiche Treppe bis zum Ende einer Stufe, danach 1.000 Epochen Sollwert 0,90. Das Training
ist deterministisch (§13: β bis Epoche 595 identisch mit Block A), also ist jeder Abzweig bis zur Abzweigung identisch mit
der Treppe; das wird unten geprüft. Abzweige nach 0,75 (Epoche 3.100), 0,65 (4.100) und 0,55 (5.100), jeweils
1.000 Epochen zurück auf 0,90. Seed 17 zusammen ≈ 20.400 Epochen ≈ 10 h.

**Vergleich:** `H_T0.900_k20_e2000` (direkt auf 0,9). **Comeback** heißt: in den letzten 200 Epochen mindestens 80 % der
Messungen von T(μ) im Band 0,9 ± 0,01. Zusätzlich berichtet: E[T(z)], R_NX(μ) bei k = 200 (globale Struktur), Krebs-AUC
auf z (nur Bewertung) und zurückgekommene Dimensionen.

Snapshots alle 100 Epochen (statt 20), damit bei 6.100 Epochen höchstens 61 Checkpoints (≈ 4 GB) im Arbeitsspeicher
liegen; die Rate und die toten Dimensionen stehen trotzdem in jeder Epoche im Controller-Log. Zuerst Seed 17; weitere
Seeds über `STAIR_SEEDS`.

In [14]:
STAIR_LEVELS = (0.95, 0.85, 0.75, 0.65, 0.55)
STAIR_LEVEL_EPOCHS, STAIR_RETURN_TARGET, STAIR_RETURN_EPOCHS = 1000, 0.90, 1000
STAIR_BRANCHES = (0.75, 0.65, 0.55)
STAIR_SEEDS = (17, 23, 31)
STAIR_SNAPSHOT_EVERY = 100
# Gewichte: Flaschenhals (μ-/logvar-Kopf, erste Decoder-Schicht) jede Epoche, ganzes Netz alle 100 Epochen.
STAIR_WEIGHTS = dict(dense=("_mu.", "_logvar.", "_decoder.0."), full_every=STAIR_SNAPSHOT_EVERY)
RUN_STAIR_TRAINING = True
STAIR_TRAINING = dict(TRAINING, checkpoint_interval=STAIR_SNAPSHOT_EVERY,
                      snapshots=dict(enabled=True, every=STAIR_SNAPSHOT_EVERY))


def level_end(level):
    """Letzte Epoche der Treppenstufe mit Sollwert `level`."""
    return CONTROLLER["warmup_epochs"] + (STAIR_LEVELS.index(level) + 1) * STAIR_LEVEL_EPOCHS


@dataclass(frozen=True)
class StairSpec:
    seed: int
    branch: float | None = None  # None: ganze Treppe

    @property
    def epochs(self):
        return level_end(STAIR_LEVELS[-1]) if self.branch is None else level_end(self.branch) + STAIR_RETURN_EPOCHS

    @property
    def schedule(self):
        levels = STAIR_LEVELS if self.branch is None else STAIR_LEVELS[:STAIR_LEVELS.index(self.branch) + 1]
        steps = [(0, levels[0])] + [(level_end(previous), level) for previous, level in zip(levels, levels[1:])]
        if self.branch is not None:
            steps.append((level_end(self.branch), STAIR_RETURN_TARGET))
        return tuple(steps)

    @property
    def name(self):
        if self.branch is None:
            return f"T_stair_to{STAIR_LEVELS[-1]:.2f}_s{self.seed}"
        return f"T_stair_b{self.branch:.2f}to{STAIR_RETURN_TARGET:.2f}_s{self.seed}"


# Test: Zeitplan der Treppe und eines Abzweigs
probe = StairSpec(17, 0.65)
assert probe.schedule == ((0, .95), (1100, .85), (2100, .75), (3100, .65), (4100, .90))
assert probe.epochs == 5100 and StairSpec(17).epochs == 5100 and StairSpec(17, 0.55).epochs == 6100
annealer = SwitchingMaximinAnnealer([Constraint("t", None, .95, .01)], probe.schedule, **CONTROLLER)
targets = {done: annealer.target_at(done) for done in (101, 1100, 1101, 4100, 4101)}
assert targets == {101: .95, 1100: .95, 1101: .85, 4100: .65, 4101: .90}, targets
print("Zeitplan geprüft:", StairSpec(17).schedule)

stair_grid = [StairSpec(seed) for seed in STAIR_SEEDS] + [StairSpec(seed, branch) for seed in STAIR_SEEDS
                                                          for branch in STAIR_BRANCHES]
stair_handle = display(Markdown("_Noch keine Messung._"), display_id=True)


def stair_live_plot(spec):
    def draw(log):
        measured = log.dropna(subset=["metric_value"])
        fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", color="black")
        axes[0].set(yscale="log", title="β/d", xlabel="Epoche")
        axes[1].plot(measured.epoch, measured.metric_value, "o-", ms=2)
        axes[1].plot(measured.epoch, measured.target, color="grey", lw=.8)
        axes[1].set(title="T(μ) und Sollwert", xlabel="Epoche")
        axes[2].plot(log.epoch, log.rate, lw=1.2)
        axes[2].set(yscale="log", title="Rate", xlabel="Epoche")
        axes[3].plot(log.epoch, log.collapsed_dimensions, lw=1.2)
        axes[3].set(title="kollabierte Dimensionen (ε-δ)", xlabel="Epoche")
        fig.suptitle(spec.name)
        fig.tight_layout()
        stair_handle.update(fig)
        plt.close(fig)
    return draw


def train_stair(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    annealer = SwitchingMaximinAnnealer([trustworthiness_constraint(STAIR_LEVELS[0], 20)], spec.schedule, **CONTROLLER)
    config = build_varix_config(data_config, "tcga", dict(STAIR_TRAINING, epochs=spec.epochs),
                                dict(RUN_DEFAULTS, beta=BETA_OVER_D_MAX * LATENT_DIM), global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, schedule=spec.schedule, annealer="SwitchingMaximinAnnealer", beta_over_d_max=BETA_OVER_D_MAX,
        controller=CONTROLLER, n_subset=N_SUBSET, subset_seed=SUBSET_SEED, epochs=spec.epochs,
        snapshot_every=STAIR_SNAPSHOT_EVERY), indent=2, default=float) + "\n")
    loss_type = annealed_varix_loss(annealer, folder, VALID_IDS, POSITIONS, on_measure=stair_live_plot(spec),
                                    weights=STAIR_WEIGHTS)
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=STAIR_SNAPSHOT_EVERY, downstream=False)
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, epochs=spec.epochs,
                                                     train_seconds=trained, total_seconds=time.time() - started),
                                                default=float) + "\n")


if RUN_STAIR_TRAINING:
    for spec in stair_grid:  # erst die Treppe, dann die Abzweige von flach nach tief
        if not (RUNS / spec.name / "run.json").exists():
            print(f"{time.strftime('%H:%M')} Lauf {spec.name} ({spec.epochs} Epochen)", flush=True)
            train_stair(spec)
stair_done = [spec for spec in stair_grid if (RUNS / spec.name / "run.json").exists()]
print("fertig:", len(stair_done), "von", len(stair_grid))

Zeitplan geprüft: ((0, 0.95), (1100, 0.85), (2100, 0.75), (3100, 0.65), (4100, 0.55))


_Noch keine Messung._

fertig: 4 von 4


### 14.1 Auswertung

Gleiche Metriken wie §13.3 (`reaction_metrics`), an jedem Snapshot (alle 100 Epochen). Determinismus: β jedes Abzweigs bis
zu seiner Abzweigung gleich der Treppe.

In [15]:
def analyse_stair(spec):
    folder = RUNS / spec.name
    cache = folder / "analysis_reaction.csv"
    if cache.exists():
        return pd.read_csv(cache)
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    table = pd.DataFrame([reaction_metrics(folder, epoch, x, reference_ids)
                          for epoch in range(STAIR_SNAPSHOT_EVERY, spec.epochs + 1, STAIR_SNAPSHOT_EVERY)])
    table.to_csv(cache, index=False)
    return table


stair = {(spec.seed, spec.branch): analyse_stair(spec) for spec in stair_done}
stair_logs = {(spec.seed, spec.branch): pd.read_csv(RUNS / spec.name / "controller_log.csv") for spec in stair_done}
for spec in stair_done:
    if spec.branch is not None and (spec.seed, None) in stair_logs:
        cut = level_end(spec.branch)
        a = stair_logs[(spec.seed, None)].set_index("epoch").effective_beta.loc[:cut]
        b = stair_logs[(spec.seed, spec.branch)].set_index("epoch").effective_beta.loc[:cut]
        print(f"{spec.name}: β bis Epoche {cut} identisch mit der Treppe: {np.allclose(a, b, rtol=1e-12, atol=0)}")

T_stair_b0.75to0.90_s17: β bis Epoche 3100 identisch mit der Treppe: True
T_stair_b0.65to0.90_s17: β bis Epoche 4100 identisch mit der Treppe: True
T_stair_b0.55to0.90_s17: β bis Epoche 5100 identisch mit der Treppe: True


**Bild 1 — Dimensionen fallen mit der Trustworthiness.** Treppe, Werte am Ende jeder Stufe: lebende Dimensionen,
gemessenes T(μ), E[T(z)], R_NX(μ) bei k = 10 und 200, Rate und Krebs-AUC gegen den Sollwert.

In [16]:
level_rows = []
for (seed, branch), table in stair.items():
    if branch is not None:
        continue
    table = table.set_index("epoch")
    for level in STAIR_LEVELS:
        end = level_end(level)
        row = table.loc[end]
        level_rows.append(dict(seed=seed, Sollwert=level, Epoche=end, lebend=LATENT_DIM - int(row.dead), T_mu=row.t_mu,
                               T_z=row.t_z, R_NX_mu_k10=row.rnx_mu_k10, R_NX_mu_k200=row.rnx_mu_k200, Rate=row.rate,
                               AUC_Krebs=row.auc_cancer, Gini=row.gini, D_eff=row.d_eff))
levels = pd.DataFrame(level_rows)
levels.to_csv(OUTPUT / "stair_levels.csv", index=False)
fig, axes = plt.subplots(1, 4, figsize=(21, 4.2))
for seed, part in levels.groupby("seed"):
    color = seed_colors.get(seed, "k")
    axes[0].plot(part.Sollwert, part.lebend, "o-", color=color, label=f"s{seed}")
    for column, style in (("T_mu", "-"), ("T_z", "--")):
        axes[1].plot(part.Sollwert, part[column], "o" + style, color=color, ms=4, label=f"{column}, s{seed}")
    for column, style in (("R_NX_mu_k10", "-"), ("R_NX_mu_k200", "--")):
        axes[2].plot(part.Sollwert, part[column], "o" + style, color=color, ms=4, label=f"{column}, s{seed}")
    axes[3].plot(part.Sollwert, part.AUC_Krebs, "o-", color=color, label=f"AUC Krebsart, s{seed}")
    rate_axis = axes[3].twinx()
    rate_axis.plot(part.Sollwert, part.Rate, "s:", color="grey", ms=3)
    rate_axis.set(yscale="log", ylabel="Rate (Nats, grau)")
axes[0].set(title="lebende Dimensionen (von 8)", ylim=(-.3, 8.3))
axes[1].set(title="Trustworthiness: T(μ) durchgezogen, E[T(z)] gestrichelt")
axes[2].set(title="R_NX(μ): k = 10 durchgezogen, k = 200 gestrichelt")
axes[3].set(title="Krebs-AUC (z) und Rate")
for axis in axes:
    axis.set_xlabel("Sollwert T(μ) der Stufe")
    axis.invert_xaxis()
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "stair_dimensions_vs_t.png", dpi=150)
plt.show()
display(levels.set_index(["seed", "Sollwert"]).style.format(precision=3))

**Bild 2 — Comeback je Abzweig.** Verläufe der Abzweige (farbig) über der Treppe (grau) und dem direkten 0,9-Lauf
(schwarz gepunktet).

In [17]:
panels = [("t_mu", "T(μ)"), ("dead", "tote Dimensionen"), ("t_z", "E[T(z)]"), ("rnx_mu_k200", "R_NX(μ), k = 200"),
          ("rnx_z_k10", "R_NX(z), k = 10"), ("rate", "Rate (Nats)"), ("gini", "Gini ganze μ-Matrix"), ("auc_cancer", "AUC Krebsart (z)")]
branch_colors = dict(zip(STAIR_BRANCHES, plt.cm.plasma(np.linspace(.1, .85, len(STAIR_BRANCHES)))))
for seed in STAIR_SEEDS:
    fig, axes = plt.subplots(2, 4, figsize=(22, 8.5))
    reference = analyse_reaction(direct_name(seed), DIRECT_EPOCHS) if (RUNS / direct_name(seed) / "run.json").exists() else None
    for axis, (column, title) in zip(axes.ravel(), panels):
        if (seed, None) in stair:
            axis.plot(stair[(seed, None)].epoch, stair[(seed, None)][column], color="grey", lw=2, alpha=.6, label="Treppe")
        for branch in STAIR_BRANCHES:
            if (seed, branch) in stair:
                table = stair[(seed, branch)]
                table = table[table.epoch >= level_end(branch)]
                axis.plot(table.epoch, table[column], color=branch_colors[branch], lw=1.2, label=f"ab {branch:.2f} → 0,90")
                axis.axvline(level_end(branch), color=branch_colors[branch], lw=.5, ls=":")
        if reference is not None:
            axis.plot(reference.epoch, reference[column], "k:", lw=.9, label="direkt 0,90")
        axis.set(title=title, xlabel="Epoche")
        if column == "rate":
            axis.set_yscale("log")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(f"Treppe und Abzweige, Seed {seed}")
    fig.tight_layout()
    fig.savefig(OUTPUT / f"stair_branches_s{seed}.png", dpi=150)
    plt.show()

In [18]:
comeback_rows = []
for spec in stair_done:
    if spec.branch is None:
        continue
    table = stair[(spec.seed, spec.branch)].set_index("epoch")
    at_branch, end = table.loc[level_end(spec.branch)], table.iloc[-1]
    log = stair_logs[(spec.seed, spec.branch)]
    late = log[log.epoch > spec.epochs - 200].dropna(subset=["metric_value"])
    in_band = float(((late.metric_value - STAIR_RETURN_TARGET).abs() <= .01).mean())
    comeback_rows.append(dict(seed=spec.seed, Abzweig=spec.branch, lebend_beim_Abzweig=LATENT_DIM - int(at_branch.dead),
                              Rate_beim_Abzweig=at_branch.rate, lebend_am_Ende=LATENT_DIM - int(end.dead),
                              T_mu_im_Band=in_band, Comeback=in_band >= .8, T_mu_Ende=end.t_mu, T_z_Ende=end.t_z,
                              R_NX_mu_k200_Ende=end.rnx_mu_k200, AUC_Krebs_Ende=end.auc_cancer,
                              beta_over_d_Ende=log.beta_over_d.iloc[-1]))
for seed in STAIR_SEEDS:
    if (RUNS / direct_name(seed) / "run.json").exists():
        end = analyse_reaction(direct_name(seed), DIRECT_EPOCHS).iloc[-1]
        comeback_rows.append(dict(seed=seed, Abzweig="direkt 0,90", lebend_am_Ende=LATENT_DIM - int(end.dead), T_mu_Ende=end.t_mu,
                                  T_z_Ende=end.t_z, R_NX_mu_k200_Ende=end.rnx_mu_k200, AUC_Krebs_Ende=end.auc_cancer))
comeback = pd.DataFrame(comeback_rows)
comeback.to_csv(OUTPUT / "stair_comeback.csv", index=False)
display(comeback.set_index(["seed", "Abzweig"]).style.format(precision=3, na_rep="")
        .format("{:.2e}", subset=["beta_over_d_Ende"], na_rep="")
        .set_caption("Comeback = in den letzten 200 Epochen ≥ 80 % der T(μ)-Messungen im Band 0,9 ± 0,01"))
reached = comeback[comeback.Comeback.eq(True)] if "Comeback" in comeback else comeback.iloc[:0]
if len(reached):
    deepest = reached.loc[reached.Rate_beim_Abzweig.idxmin()]
    print(f"Tiefster Abzweig mit Comeback: ab T = {deepest.Abzweig:.2f}, {int(deepest.lebend_beim_Abzweig)} lebende "
          f"Dimensionen, Rate {deepest.Rate_beim_Abzweig:.2f} Nats beim Abzweig")

Tiefster Abzweig mit Comeback: ab T = 0.65, 2 lebende Dimensionen, Rate 0.33 Nats beim Abzweig


### 14.2 Befunde (gemessen, Seed 17, 1.000 Epochen pro Stufe; Tabellen und Bilder oben)

**Determinismus.** β jedes Abzweigs ist bis zur Abzweigung identisch mit der Treppe (Zelle 14.1).

**1. Dimensionen fallen stufenweise mit dem Sollwert** (Ende jeder Stufe, lebend nach R_j-Kriterium):

| Sollwert T(μ) | T(μ) erreicht | lebend | Rate (Nats) | Krebs-AUC (z) | R_NX(μ) k = 200 |
|---|---|---|---|---|---|
| 0,95 | 0,951 | 8 | 15,0 | 0,96 | 0,52 |
| 0,85 | 0,855 | 7 | 1,92 | 0,75 | 0,47 |
| 0,75 | 0,805 | 3 | 2,57 | 0,72 | 0,38 |
| 0,65 | 0,640 | 2 | 0,33 | 0,56 | 0,22 |
| 0,55 | nicht berechenbar | 0 | 2·10⁻⁴ (Boden) | 0,51 | – |

- Bei 0,75 wird der Sollwert verfehlt (0,805) und die Rate ist höher als bei 0,85. Grund (im Verlauf abgelesen): beim
  Übergang fallen 5 Dimensionen auf einmal, T stürzt auf ≈ 0,70 unter das Band, der Controller nimmt β zurück
  (β/d 1,7·10⁻³ → 0,9·10⁻³), die 3 übrigen Dimensionen bekommen mehr Rate. Der Collapse-Sprung überschießt den Sollwert.
- Bei 0,55 ist der Collapse vollständig: Epoche 4.200–4.600 ist μ für jeden Patienten exakt 0 (AUTOENCODIX speichert `mu`
  dann nicht, siehe `load_snapshot`). D_eff = 8,0 dort ist ein Artefakt: z ist reines isotropes Rauschen.

**2. Comeback auf T(μ) = 0,9** (Kriterium: ≥ 80 % der T-Messungen der letzten 200 Epochen im Band 0,9 ± 0,01):

| Abzweig | lebend beim Abzweig | lebend am Ende | Comeback | T(μ) Ende | Rate Ende | Krebs-AUC Ende | R_NX(μ) k = 200 |
|---|---|---|---|---|---|---|---|
| 0,75 | 3 | 4 | ja | 0,899 | 9,1 | 0,87 | 0,43 |
| 0,65 | 2 | 4 | ja | 0,901 | 8,5 | 0,87 | 0,44 |
| 0,55 | 0 | 0 | **nein** | 0,771 | 1·10⁻³ | 0,52 | 0,30 |
| direkt 0,9 | – | 8 | – | 0,902 | 3,0–3,5 | 0,83 | 0,51 |

- **Tiefster Zustand mit Comeback: 2 lebende Dimensionen** (Abzweig 0,65, Rate 0,33 Nats). Ob 1 lebende Dimension reicht,
  deckt das Gitter nicht ab.
- Die Rückkehrer landen beide bei 4 lebenden Dimensionen, mit fast dreifacher Rate des direkten Laufs (Hysterese wie §13)
  und **höherer** Krebs-AUC (0,87 gegen 0,83), aber kleinerem R_NX(μ) bei k = 200 (global nicht voll zurück, wie §13).

**3. Vollständiger Collapse: nicht strikt irreversibel, praktisch aber kein Weg zurück.**
- Ab Epoche ≈ 4.500 liegt β bei 0 (Controller am Anschlag). Ab Epoche 4.700 bekommen erste Trainingspatienten wieder μ > 0,
  am Ende 92 % der Patienten (in 3–4 Dimensionen). Die Herleitung „alle μ auf der Klemme ⇒ kein Gradient ⇒ kein Weg
  zurück“ stimmt für Varix in dieser Konfiguration also **nicht**.
- Trotzdem nach 1.000 Epochen bei β = 0: T(μ) 0,77 (zwischen Epoche 5.900 und 6.100 nahezu flach), Rate 1·10⁻³ Nats,
  max μ ≈ 0,05 neben σ ≥ 1,005 (logvar-Klemme), Krebs-AUC 0,52 = Zufall. Die Struktur auf μ kehrt teilweise zurück
  (T ist rangbasiert und skalenblind), die Information in z nicht.
- **Hypothese** zum Ausweg: Weight Decay (0,01) zieht die Gewichte gegen 0, negative Vor-Klemm-Werte von μ wandern über die
  Schwelle 10⁻⁷, und der Gradient ist zurück. Dafür spricht die Verzögerung von ≈ 200 Epochen nach β = 0. Test: derselbe
  Abzweig mit wd = 0.

**Grenzen.** Ein Seed; Gitter mit Abstand 0,1; 1.000 Epochen Rückkehr (ob T bei 0,55 mit mehr Epochen 0,9 erreicht, offen).
